# WHM — paper-figure notebook

Reproduces the figures of Brieden et al. (2025), *The web-halo model* (arXiv:2508.10902): Fig. 2 (`halomassfunctions.png`), Fig. 4 (`WHMcode_spectra_comp_kNL1_concnfw_narya.png` + legend), Fig. 5 (`w0waMnu_LPT_kNL1.0_BACCO_baselineaccuratebf_concnfwwarren_100.png`), Figs. 7/8 (`theo_vs_measured_p1h_linlag_all*.png`, `HMF_cumulative_integrand_comparison.png`) and Fig. A1 (`test_cutoff_and_jn.png`). Fig. 3 is made by `whmcode_windowfunctions.ipynb`, Figs. 6 and A2 by `WHMcode_LPT_scatter.ipynb` (which reads the per-cosmology spectra written by sections 5-7 of this notebook).

Runs from the repository root. Inputs live under `data/`, figures are written to `figures/`. Everything is regenerated from scratch when the `RECOMPUTE` flags are `True`; with `WRITE=True` the results are cached under `data/` and reloaded on the next run. The cosmology sample used in sections 4-7 is selected with `SAMPLE` (`BACCO`, `EE2` or `CosmicEmu`); the configuration cell below documents the environment variables that override these settings for headless runs.

Requires the modified `WHM-CAMB` from https://github.com/sbrieden/WHM (with the `WHM_hmf` / `WHM_thinweb` switches), `classy` (CLASS 3.3), `velocileptors`, `baccoemu`, `euclidemu2` and `FlamingoBaryonResponseEmulator`; see `env/`.


In [ ]:
import euclidemu2 as ee2
import warnings
warnings.filterwarnings('ignore')
import os
import sys
import numpy as np
import scipy
import camb
import matplotlib
import matplotlib.pyplot as plt
import math
from random import random
import time
import baccoemu
from classy import Class
from velocileptors.LPT.lpt_rsd_fftw import LPT_RSD
from velocileptors.LPT.moment_expansion_fftw import MomentExpansion

# --- euclidemu2 >= 1.4 compatibility shim --------------------------------
# Newer euclidemu2 replaced the module-level get_pnonlin(params, z, k) by
# PyEuclidEmulator.get_boost / get_plin; P_NL = boost x P_lin, with the boost
# constant-extrapolated below k_min = 8.73e-3 h/Mpc. The linear P(k) under the
# boost is NOT taken from get_plin: euclidemu2 1.4.3 computes it with three
# degenerate massive neutrinos (N_ncdm=3, N_ur=0.0044), whereas EuclidEmulator2
# and the euclidemu2 <= 1.2 used for the paper assume ONE massive species
# (N_ncdm=1, N_ur=2.03). It is computed here with classy in the neutrino
# convention of the current sample (SAMPLE_CFG: N_ur_massive / ncdm_deg, i.e.
# 2.0328 / 1 for the EE2 sample), otherwise exactly as euclidemu2 <= 1.2 did:
# Omega_cdm = Omega_m - Omega_b - Sum m_nu/93.14/h^2, total-matter P(k) in (Mpc/h)^3.
# Old param keys (Omm/Omb/mnu/w/As/ns/h/wa) are mapped to the new ones.
_ee2_emulator = ee2.PyEuclidEmulator()
def _ee2_plin(p, custom_k, zlist):
    """Linear total-matter P(k) [(Mpc/h)^3 on custom_k in h/Mpc] for the EE2 boost."""
    h = p['h']
    pars = {'h': h, 'Omega_b': p['Omega_b'], 'n_s': p['n_s'], 'A_s': p['A_s'],
            'Omega_cdm': p['Omega_m'] - p['Omega_b'] - p['m_ncdm']/93.14/h**2,
            'Omega_Lambda': 0.0, 'w0_fld': p['w0_fld'], 'wa_fld': p['wa_fld'],
            'output': 'mPk', 'P_k_max_h/Mpc': 1.05*float(np.max(custom_k)),
            'z_max_pk': float(max(zlist)) + 0.5}
    if p['m_ncdm'] > 1e-5:
        pars.update({'N_ur': CFG['N_ur_massive'], 'N_ncdm': 1, 'deg_ncdm': CFG['ncdm_deg'],
                     'm_ncdm': p['m_ncdm']/CFG['ncdm_deg']})
    else:
        pars.update({'N_ur': CFG['N_ur_massless']})
    c = Class(); c.set(pars); c.compute()
    plin = {j: np.array([c.pk_lin(k*h, z)*h**3 for k in custom_k]) for j, z in enumerate(zlist)}
    c.struct_cleanup(); c.empty()
    return plin
def _ee2_get_pnonlin(params, redshifts, custom_k):
    p = {'Omega_b': params['Omb'], 'Omega_m': params['Omm'],
         'm_ncdm': params.get('mnu', 0.06), 'n_s': params['ns'],
         'h': params['h'], 'w0_fld': params.get('w', -1.0),
         'wa_fld': params.get('wa', 0.0), 'A_s': params['As']}
    zlist = [float(z) for z in np.atleast_1d(redshifts)]
    custom_k = np.asarray(custom_k, dtype=float)
    try:
        _, boost = _ee2_emulator.get_boost(p, zlist, custom_k)
        plin = _ee2_plin(p, custom_k, zlist)
    except Exception as exc:
        # Cosmology outside the EE2 emulator range (e.g. a BACCO-sample point
        # with Sum m_nu > 0.15 eV or h > 0.73): return NaNs so the primary run
        # does not crash. EE2 figures require the dedicated EE2 cosmology sample.
        if not getattr(_ee2_get_pnonlin, '_warned', False):
            print('  [EE2 shim] emulator/classy failure -> NaN:', str(exc).strip()[:80])
            _ee2_get_pnonlin._warned = True
        nan = {j: np.full_like(custom_k, np.nan) for j in range(len(zlist))}
        return custom_k, dict(nan), dict(nan), dict(nan)
    boost = {j: np.asarray(boost[j]) for j in boost}
    pnl = {j: plin[j] * boost[j] for j in plin}
    return custom_k, pnl, plin, boost
ee2.get_pnonlin = _ee2_get_pnonlin

# --- HMcode displacement scale sigma_v (missing from standard classy) -------
# The original notebook used a modified CLASS exposing fourier_hmcode_sigmadisp
# [_cb][100]; standard classy 3.3.4 lacks them and its C type is immutable, so we
# subclass and rebind `Class`. sigma_v^2 = (1/(6 pi^2)) \int dk P_lin(k) (physical
# Mpc; k in 1/Mpc, P in Mpc^3),
# the _cb variant on the cold (cdm+baryon) spectrum, and an R=100 Mpc/h top-hat
# variant. Returned in Mpc/h (so 1/sigma_v is in h/Mpc, like nonlinear_scale).
_BaseClass = Class
_sigmadisp_cache = {}
class Class(_BaseClass):
    def _hmcode_sigmadisp(self, z, nz=None, cold=False, R=0.0, npts=600):
        zs = tuple(float(x) for x in (np.atleast_1d(z)[:nz] if nz is not None else np.atleast_1d(z)))
        key = (id(self), zs, cold, R)
        if key in _sigmadisp_cache:
            return _sigmadisp_cache[key]
        # Physical (Mpc) units to match the paper convention: k in 1/Mpc, P in
        # Mpc^3, so sigma_v is in Mpc and kv = 1/sigma_v in 1/Mpc. Validated to
        # <0.1% against data/knl/kv_fid.txt (cold) and kv_w0wamnu_BACCO.txt (total)
        # across the full 100-cosmology sample.
        kk = np.logspace(-4, np.log10(45.0), npts)   # 1/Mpc
        pkf = self.pk_cb_lin if cold else self.pk_lin
        out = []
        for zz in zs:
            try:
                P = np.array([pkf(k, zz) for k in kk])      # Mpc^3
            except Exception:
                P = np.array([self.pk_lin(k, zz) for k in kk])
            integ = P
            if R > 0:
                x = kk*R; W = 3*(np.sin(x)-x*np.cos(x))/x**3; integ = P*W**2
            out.append(np.sqrt(np.trapezoid(integ, kk)/(6*np.pi**2)))
        res = np.array(out)
        _sigmadisp_cache[key] = res
        return res
    def fourier_hmcode_sigmadisp(self, z, nz=None):
        return self._hmcode_sigmadisp(z, nz, cold=False, R=0.0)
    def fourier_hmcode_sigmadisp_cb(self, z, nz=None):
        return self._hmcode_sigmadisp(z, nz, cold=True, R=0.0)
    def fourier_hmcode_sigmadisp100(self, z, nz=None):
        return self._hmcode_sigmadisp(z, nz, cold=False, R=100.0)


In [ ]:
import os

# --- project paths (relative to the project root = cwd) ---
DATA_DIR = "data"
FIG_DIR  = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

# --- per-source read / compute / write control ---
# keys: camb, velocileptors, bacco, ee2, miratitan, nbody
# RECOMPUTE[k]=False -> load cached results when present; True -> recompute.
# WRITE[k]=True      -> persist (savetxt/np.save) computed results to data/.
RECOMPUTE = {"camb": False, "velocileptors": False, "bacco": False,
             "ee2": False, "miratitan": False, "nbody": False}
WRITE     = {"camb": False, "velocileptors": False, "bacco": False,
             "ee2": False, "miratitan": False, "nbody": False}
# Environment overrides for headless runs: WHM_RECOMPUTE / WHM_WRITE = "all" or a comma list of the keys above,
# WHM_SAMPLE = BACCO | EE2 | CosmicEmu, WHM_NCOSMO = number of sample cosmologies.
_rc = os.environ.get("WHM_RECOMPUTE", ""); _wr = os.environ.get("WHM_WRITE", "")
for _k in RECOMPUTE: RECOMPUTE[_k] = RECOMPUTE[_k] or (_rc == "all") or (_k in _rc.split(","))
for _k in WRITE:     WRITE[_k]     = WRITE[_k]     or (_wr == "all") or (_k in _wr.split(","))

def load_or_compute(path, compute_fn, recompute=False, write=False,
                    loader=np.loadtxt, saver=np.savetxt, **load_kwargs):
    """Cache-aware loader.

    Load ``path`` with ``loader`` if it exists and ``recompute`` is False;
    otherwise call ``compute_fn()``, optionally persist it with ``saver`` when
    ``write`` is True, and return it. Use loader=np.load / saver=np.save for
    .npy files; pass loadtxt kwargs (e.g. unpack=True) via **load_kwargs.
    """
    if (not recompute) and os.path.exists(path):
        return loader(path, **load_kwargs)
    result = compute_fn()
    if write:
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        saver(path, result)
    return result

# --- make np.savetxt/np.save create parent directories on demand -----------
# Many gated `if WRITE[...]: np.savetxt('data/<sub>/...')` sites write into
# cache subdirs that may not exist yet; numpy does not mkdir them. Wrap the two
# writers so any write auto-creates its parent directory.
import numpy as _np
def _mkdir_for(path):
    d = os.path.dirname(str(path))
    if d:
        os.makedirs(d, exist_ok=True)
if not getattr(_np.savetxt, "_mkdir_wrapped", False):
    _orig_savetxt = _np.savetxt
    def savetxt(fname, *a, **k):
        _mkdir_for(fname); return _orig_savetxt(fname, *a, **k)
    savetxt._mkdir_wrapped = True
    _np.savetxt = savetxt
if not getattr(_np.save, "_mkdir_wrapped", False):
    _orig_save = _np.save
    def save(file, *a, **k):
        _mkdir_for(file); return _orig_save(file, *a, **k)
    save._mkdir_wrapped = True
    _np.save = save

# --- cosmology sample for the per-cosmology sections 4-7 (Figs. 5, 6, A2) ---
# BACCO (paper default): 100 cosmologies in the baccoemu range; EE2: 100 in the EuclidEmulator2 range;
# CosmicEmu: the 112 Mira-Titan models (spectra in Mpc units, k up to 5 Mpc^-1, as provided by the emulator).
# N_ur_massive / N_ur_massless / ncdm_deg: neutrino convention of the sample's CLASS models (camb_nu_*: the same
#   for the CAMB runs). BACCO and CosmicEmu: three degenerate massive species (N_eff 3.044 / 3.04); EE2: ONE massive
#   species with N_ur = 2.0328, which is also the convention of the linear P(k) that the EuclidEmulator2 boost multiplies
#   (see the euclidemu2 shim in the first cell) - using three species there shifts the EE2 row of Fig. 6 by ~0.5% at low k.
SAMPLE = os.environ.get("WHM_SAMPLE", "BACCO")
SAMPLE_CFG = {
    "BACCO":     dict(Ncosmo=100, suffix="w0wamnu_BACCO",           kmax=4.8,  hubble_units=True,  class_norm="sigma8",
                      N_ur_massive=0.00441, N_ur_massless=3.044, ncdm_deg=3, camb_nu_massless=0.00441, camb_nu_deg=3.03959),
    "EE2":       dict(Ncosmo=100, suffix="w0wamnu_EE2",             kmax=9.41, hubble_units=True,  class_norm="A_s",
                      N_ur_massive=2.0328,  N_ur_massless=3.044, ncdm_deg=1, camb_nu_massless=2.0328,  camb_nu_deg=1.0112),
    "CosmicEmu": dict(Ncosmo=112, suffix="w0wamnu_CosmicEmu_1_Mpc", kmax=5.0,  hubble_units=False, class_norm="sigma8",
                      N_ur_massive=0.00041, N_ur_massless=3.04,  ncdm_deg=3, camb_nu_massless=0.00041, camb_nu_deg=3.03959),
}
assert SAMPLE in SAMPLE_CFG, SAMPLE
CFG = SAMPLE_CFG[SAMPLE]
# --- velocileptors (1-loop LPT) FFTLog grid -------------------------------------------------------------
# The paper's LPT spectra were computed with velocileptors as of July 2025 (commit 83d82ae, 2025-07-10), whose
# LPT_RSD defaults were N=2000 points over 10^-5 .. 10^3 /Mpc (extrap_min=-5, extrap_max=3). Commit f8dae25
# (2026-04-23, "Replaced old module", released as velocileptors 3.1) replaced LPT/lpt_rsd_fftw.py by an optimized
# rewrite whose defaults are N=256 points over 10^-4 .. 10 /Mpc (extrap_min=-4, extrap_max=1). With those defaults
# the spectra differ from the paper's by 1-3% below k = 1 h/Mpc and by up to 20% of P_NL beyond the cutoff
# k_c = sqrt(5) k_nl (largest for low-sigma8 cosmologies at z = 1.5; visible as a wider outer band in the
# EuclidEmu2 row of Fig. 6). Passing the July-2025 values explicitly reproduces the paper's spectra to < 1e-3
# with velocileptors 3.1 (each call ~8x slower than with the 3.1 defaults). Used by every LPT_RSD call below.
LPT_GRID = dict(N=2000, extrap_min=-5, extrap_max=3)

print("sample:", SAMPLE, "| RECOMPUTE:", RECOMPUTE, "| WRITE:", WRITE)


## 1. Cosmology parameters & sampling

In [ ]:
om_m, om_b, s8, hubble, n_s, w0, wa, om_nu = np.loadtxt('data/CosmicEmu/MiraTitanParams.dat', unpack=True)

om_cdm = om_m-om_b-om_nu   # physical densities omega_x; the fractional Omega_x are derived in the next cells
Om_cb = (om_cdm+om_b)/hubble**2
Om_b = om_b/hubble**2

if WRITE["miratitan"]: np.savetxt('data/cosmo/hubble_w0wamnu_CosmicEmu.txt',hubble)
if WRITE["miratitan"]: np.savetxt('data/cosmo/om_cdm_w0wamnu_CosmicEmu.txt',om_cdm)
if WRITE["miratitan"]: np.savetxt('data/cosmo/om_b_w0wamnu_CosmicEmu.txt',om_b)
if WRITE["miratitan"]: np.savetxt('data/cosmo/om_nu_w0wamnu_CosmicEmu.txt',om_nu)
if WRITE["miratitan"]: np.savetxt('data/cosmo/n_s_w0wamnu_CosmicEmu.txt',n_s)
if WRITE["miratitan"]: np.savetxt('data/cosmo/s8_w0wamnu_CosmicEmu.txt',s8)
if WRITE["miratitan"]: np.savetxt('data/cosmo/w0_w0wamnu_CosmicEmu.txt',w0)
if WRITE["miratitan"]: np.savetxt('data/cosmo/wa_w0wamnu_CosmicEmu.txt',wa)


In [ ]:
from scipy.integrate import simpson
from scipy import interpolate

def correlation_to_power_spectrum(r, xi_r, k=None):
    """
    Compute the power spectrum from the correlation function using a Fourier transform,
    with wavenumbers corresponding to the input distances.
    
    Parameters:
    r (numpy array): 1D array of distances (r).
    xi_r (numpy array): 1D array of correlation function values corresponding to r.
    
    Returns:
    k (numpy array): 1D array of wavenumbers corresponding to the power spectrum.
    P_k (numpy array): 1D array of the power spectrum values corresponding to k.
    """
    # Define the wavenumbers k as corresponding to input distances r
    if k is None:
        k = sorted(2 * np.pi / r)
    
    P_k = np.zeros_like(k)
    
    # Loop over each wavenumber k and compute the Fourier transform
    for i, k_i in enumerate(k):
        integrand = r**2 * xi_r * np.sin(k_i * r) / (k_i * r)
        P_k[i] = 4 * np.pi * simpson(integrand, x=r)
    
    return k, P_k


In [ ]:
# general settings for the runs
high_precision = False # if True, the runs may take several minutes
random_cosmology = True # if True, random cosmoparams in a certain prior range are chosen, else, user defined
want_output_feedback = False # whether to print feedback of CLASS and CAMB in terminal or not
want_varying_dark_energy = True #if True, uses PPF parameterization of the DE eq. of state: w=w0+(1-a)*wa, (or equivalently w=w0wa-a*(w0wa-w0))
want_varying_neutrino_mass = True #if False, Mnu is fixed to 0.06 with 1 massive and 2 massless neutrinos. In any case Neff is fixed to 3.046
want_varying_baryonic_feedback = False # whether to vary the baryonic feedback parameters or not. By default they are fixed to the dmonly case
start_from_previous = True


In [ ]:
# Set this to True, if you want to redo a run with the same choice of cosmoparams as before. Only works with jupyter notebook
Ncosmo = int(os.environ.get("WHM_NCOSMO", CFG["Ncosmo"]))

YHe = 0.245
T_CMB = 2.7255

# Priors for Cosmological parameters for BACCO 
hubble_prior = [0.6,0.8]
Om_cb_prior = [0.24,0.4]
Om_b_prior = [0.04,0.06]
s8_prior = [0.70,0.9]
s8_cb_prior = [0.73,0.9]
n_s_prior = [0.92,1.01]

hubble_fid = 0.6736
om_cdm_fid = 0.12
om_b_fid = 0.02237
A_s_fid = 2.0830e-9
n_s_fid = 0.9649

hubble_bacco = [0.65,0.6,0.7]
om_cdm_bacco = [0.088725, 0.0954, 0.1519]
om_b_bacco = [0.02535, 0.018, 0.0245]
s8_bacco = [0.9, 0.9, 0.9]
n_s_bacco = [0.92, 1.01, 1.01]

common_settings = {
                   # output and precision parameters
                   'output':'mPk',
                   'z_pk': '0.0, 0.5, 0.8, 1.0, 1.3, 1.5',
                   'P_k_max_1/Mpc':50.0,
                   'non linear':'hmcode',
                   'hmcode_version':'2016',
                   #'non linear':'halofit',
                   #'non linear':'PT',
                   #'IR resummation':'Yes',
                   #'Bias tracers':'No',
                   #'cb':'No',
                   #'RSD':'No',
                   # fixed LambdaCDM parameters
                   'tau_reio':0.05430842,
                   'YHe':YHe,
                   'T_cmb':T_CMB,}  

# Priors for extensions to LCDM
Mnu_prior = [0.0,0.4]
w0_prior = [-1.15,-0.85] # only passed to CLASS/CAMB if use_varying_dark_energy = True
wa_prior = [-0.3,0.3] # only passed to CLASS/CAMB if use_varying_dark_energy = True

#Priors for baryonic feedback

if (start_from_previous):
    # load the cosmology sample selected by SAMPLE: data/cosmo/<parameter>_w0wamnu_<SAMPLE>.txt
    _tag = 'w0wamnu_' + SAMPLE
    def _load_param(name): return np.loadtxt('data/cosmo/%s_%s.txt' % (name, _tag))
    hubble = _load_param('hubble'); om_cdm = _load_param('om_cdm'); om_b = _load_param('om_b')
    A_s = _load_param('A_s'); n_s = _load_param('n_s')
    s8 = _load_param('s8'); Mnu = _load_param('Mnu'); w0 = _load_param('w0'); wa = _load_param('wa')
    s8_cb = _load_param('s8_cb') if SAMPLE == 'BACCO' else np.zeros(len(s8))
    M = {}
    for i in range(0,Ncosmo):
        M[i] = Class()
        M[i].set(common_settings)
        M[i].set({'omega_cdm':om_cdm[i]})
        M[i].set({'omega_b':om_b[i]})
        M[i].set({'n_s':n_s[i],})
        M[i].set({'h':hubble[i],})
        if CFG['class_norm'] == 'A_s':
            M[i].set({'A_s':A_s[i],})
        else:
            M[i].set({'sigma8':s8[i],})
        if (Mnu[i]>1e-5):
            M[i].set({'N_ur': CFG['N_ur_massive']})
            M[i].set({'N_ncdm': 1})
            M[i].set({'deg_ncdm': CFG['ncdm_deg']})
            M[i].set({'m_ncdm': Mnu[i]/CFG['ncdm_deg']})
        else:
            M[i].set({'N_ur': CFG['N_ur_massless']})
        if (w0[i]!=-1.0 or wa[i]!=0.0):
            M[i].set({'Omega_Lambda': 0.0})
            M[i].set({'w0_fld': w0[i]})
            M[i].set({'wa_fld': wa[i]})
        M[i].compute()
else:
    hubble = np.ones(Ncosmo)
    om_cdm = np.ones(Ncosmo)
    om_b = np.ones(Ncosmo)
    A_s = np.ones(Ncosmo)
    n_s = np.ones(Ncosmo)
    Om_cb = np.ones(Ncosmo)
    Om_b = np.ones(Ncosmo)
    s8 = np.ones(Ncosmo)
    s8_cb = np.ones(Ncosmo)
    Mnu = np.zeros(Ncosmo)
    w0 = -np.ones(Ncosmo)
    wa = np.zeros(Ncosmo)
    M = {}
    counter = -1
    i=-1
    while i<Ncosmo-1:
        if (random_cosmology):
            i+=1
            counter+=1
            print(counter, i)
            # choose random value within the flat prior range
            hubble[i] = hubble_prior[0] + random()*(hubble_prior[1]-hubble_prior[0])
            Om_cb[i] = Om_cb_prior[0] + random()*(Om_cb_prior[1]-Om_cb_prior[0])
            Om_b[i] = Om_b_prior[0] + random()*(Om_b_prior[1]-Om_b_prior[0])
            om_cdm[i] = (Om_cb[i]-Om_b[i])*hubble[i]**2
            om_b[i] = (Om_b[i])*hubble[i]**2
            n_s[i] = n_s_prior[0] + random()*(n_s_prior[1]-n_s_prior[0])
            s8[i] = s8_prior[0] + random()*(s8_prior[1]-s8_prior[0])
            A_s[i] = A_s_prior[0] + random()*(A_s_prior[1]-A_s_prior[0])
            M[i] = Class()
            M[i].set(common_settings)
            M[i].set({'omega_cdm':om_cdm[i]})
            M[i].set({'omega_b':om_b[i]})
            M[i].set({'n_s':n_s[i]})
            M[i].set({'h':hubble[i]})
            M[i].set({'sigma8':s8[i]})

            if (want_varying_neutrino_mass):
                Mnu[i] = Mnu_prior[0] + random()*(Mnu_prior[1]-Mnu_prior[0])
                M[i].set({'N_ur': 0.00441})
                M[i].set({'N_ncdm': 1})
                M[i].set({'deg_ncdm': 3})
                M[i].set({'m_ncdm': Mnu[i]/3.})
            if (want_varying_dark_energy):
                w0[i] = w0_prior[0] + random()*(w0_prior[1]-w0_prior[0])
                wa[i] = wa_prior[0] + random()*(wa_prior[1]-wa_prior[0])
                M[i].set({'Omega_Lambda': 0.0})
                M[i].set({'w0_fld': w0[i]})
                M[i].set({'wa_fld': wa[i]})
            if (want_varying_baryonic_feedback):
                c_min = c_min_prior[0] + random()*(c_min_prior[1]-c_min_prior[0])
                eta_0 = eta_0_prior[0] + random()*(eta_0_prior[1]-eta_0_prior[0])
            M[i].compute()
            A_s[i] = M[i].get_current_derived_parameters(['A_s'])['A_s']
            s8_cb[i] = M[i].sigma8_cb()
            print('(sigma8, sigma8_cb, Mnu)=(%.3f, %.3f, %.3f) at i=%d'%(s8[i], s8_cb[i], Mnu[i], i))
            if (s8_cb[i]<s8_cb_prior[0] or s8_cb[i]>s8_cb_prior[1]):
                print('Warning:sigma8_cb=%.2f at i=%d outside prior range'%(s8_cb[i], i))
                i-=1

        
        else:
            # put fiducial parameters manually here
            hubble[i]  = hubble_prior[0]+(hubble_prior[1]-hubble_prior[0])*i/(Ncosmo-1)
            om_cdm[i]  = om_cdm_prior[0]+(om_cdm_prior[1]-om_cdm_prior[0])*i/(Ncosmo-1)
            om_b[i]  = om_b_prior[0]+(om_b_prior[1]-om_b_prior[0])*i/(Ncosmo-1)
            A_s[i]  = A_s_prior[0]+(A_s_prior[1]-A_s_prior[0])*i/(Ncosmo-1)
            n_s[i]  = n_s_prior[0]+(n_s_prior[1]-n_s_prior[0])*i/(Ncosmo-1)
            if (want_varying_neutrino_mass):
                Mnu = 0.0
            if (want_varying_dark_energy):
                w0 = -1.
                w0wa = -1.

# Fractional densities derived from the physical omega_* (only omega_* files are stored under data/cosmo: on
# case-insensitive filesystems 'Om_b_*' and 'om_b_*' would be the same file).
Om_b  = om_b/hubble**2
Om_cb = (om_cdm+om_b)/hubble**2

# Other physical parameters, that need to be fixed to the same value in CAMB and CLASS


In [ ]:
M_bacco = {}
for i in range(3):
    M_bacco[i] = Class()
    M_bacco[i].set(common_settings)
    M_bacco[i].set({'omega_cdm':om_cdm_bacco[i]})
    M_bacco[i].set({'omega_b':om_b_bacco[i]})
    M_bacco[i].set({'n_s':n_s_bacco[i]})
    M_bacco[i].set({'h':hubble_bacco[i]})
    M_bacco[i].set({'sigma8':s8_bacco[i]})
    M_bacco[i].set({'c_min': 4.0})
    M_bacco[i].set({'eta_0': 0.0})
    M_bacco[i].compute()
    print(M_bacco[i].sigma8())


In [ ]:
params_Marcos = {
 'output':'mPk',
 'z_pk': '0.0, 0.5, 0.8, 1.0, 1.3, 1.5',
 'P_k_max_1/Mpc':50.0,
 'non linear':'hmcode',
 'hmcode_version':'2016',
 'h': 0.6766,
 'sigma8': 0.8102,
 'omega_b': 0.04897*0.6766*0.6766,
 'omega_cdm': 0.26067*0.6766*0.6766,
 'n_s': 0.9665,
 'tau_reio': 0.0561,
 'N_ur': 3.046,
 'T_cmb': 2.7255,
}

M_Marcos = Class()
M_Marcos.set(params_Marcos)
M_Marcos.compute()
print(M_Marcos.Omega_nu)


In [ ]:
M_fid = Class()
M_fid.set(common_settings)
M_fid.set({'omega_cdm':om_cdm_fid})
M_fid.set({'omega_b':om_b_fid})
M_fid.set({'n_s':n_s_fid})
M_fid.set({'h':hubble_fid})
M_fid.set({'A_s':A_s_fid})
M_fid.set({'N_ncdm': 1})
M_fid.set({'N_ur': 2.0328})
M_fid.set({'deg_ncdm': 1})
M_fid.set({'omega_ncdm': 0.00064420})
M_fid.compute()
print(M_fid.Omega_nu)
print(M_fid.Omega_nu/M_fid.h()**2)


In [ ]:
# Output format parameters that need to be fixed in both CAMB and CLASS
kmax_for_computation = 50.0 #in h/Mpc, has to be higher than kmax to avoid interpolation errors, this is passed to CAMB/CLASS
kmaxbacco = 4.8 # not passed to Boltzmann code, just used for plotting
kmaxee2 = 9.41
kmax_CE_1_Mpc = 5.0
kmax = 4.8 # not passed to Boltzmann code, just used for plotting


In [ ]:
params = {}
pk_lin_total = {}
pk_nl_total = {}
pk_lin_bacco = {}
pk_nl_bacco = {}
pk_lptbacco = {}
pkgal_lpt = {}
redshifts = np.array([0.0, 0.8, 1.5])
redshift_styles = ['-', '-', '-']
avec = 1./(1.+redshifts)

kvec = np.logspace(-3.9999999999,np.log10(kmax),100) # k in h/Mpc
kvecLPT = kvec[kvec<=0.75]
kvecbacco = kvec[kvec<=kmaxbacco]
kvecee2 = kvec[kvec<=kmaxee2]
print(kvec.shape)
print(kvecee2.shape)
print(kvecbacco.shape)
print(kvecLPT.shape)
kvec_sample = np.logspace(-3.9999999999,np.log10(CFG['kmax']),100) # k grid of the per-cosmology sample (h/Mpc, or 1/Mpc for CosmicEmu)
hubble_units_sample = CFG['hubble_units']


## 2. N-body reference spectra (Marcos)


In [ ]:
# N-body reference spectra of the Planck-2018 simulation (provided inputs under data/Pk_Marcos/)
rvec_Marcos, Xi1h_Marcos = np.loadtxt(os.path.join(DATA_DIR, 'Pk_Marcos', '1haloterm_Planck18_N1536_L512.txt'), unpack=True)
kvec_Marcos, Pklin_Marcos = np.loadtxt(os.path.join(DATA_DIR, 'Pk_Marcos', 'Pklin_Planck18_N1536_L512.txt'), unpack=True)
kvec_Marcos, Pknl_Marcos = np.loadtxt(os.path.join(DATA_DIR, 'Pk_Marcos', 'Pknl_Planck18_N1536_L512.txt'), unpack=True)
k_1h_Marcos, Pk1h_Marcos = correlation_to_power_spectrum(rvec_Marcos, Xi1h_Marcos, k=kvec_Marcos)


In [ ]:
kvec_Marcos = np.array([0.01995, 0.03985, 0.05975, 0.07965, 0.09955, 0.11945, 0.13935,
        0.15925, 0.17915, 0.19905, 0.21895, 0.23885, 0.25875, 0.27865,
        0.29855, 0.31845, 0.33835, 0.35825, 0.37815, 0.39805, 0.41795,
        0.43785, 0.45775, 0.47765, 0.49755, 0.51745, 0.53735, 0.55725,
        0.57715, 0.59705, 0.61695, 0.63685, 0.65675, 0.67665, 0.69655,
        0.71645, 0.73635, 0.75625, 0.77615, 0.79605, 0.81595, 0.83585,
        0.85575, 0.87565, 0.89555, 0.91545, 0.93535, 0.95525, 0.97515,
        0.99505, 1.01495, 1.03485, 1.05475, 1.07465, 1.09455, 1.11445,
        1.13435, 1.15425, 1.17415, 1.19405, 1.21395, 1.23385, 1.25375,
        1.27365, 1.29355, 1.31345, 1.33335, 1.35325, 1.37315, 1.39305,
        1.41295, 1.43285, 1.45275, 1.47265, 1.49255, 1.51245, 1.53235,
        1.55225, 1.57215, 1.59205, 1.61195, 1.63185, 1.65175, 1.67165,
        1.69155, 1.71145, 1.73135, 1.75125, 1.77115, 1.79105, 1.81095,
        1.83085, 1.85075, 1.87065, 1.89055, 1.91045, 1.93035, 1.95025,
        1.97015, 1.99005])
Pk1h_Marcos = np.array([550.65688097, 479.08087044, 417.75235557, 443.19107343,
        427.74013555, 434.7135989 , 428.33146314, 429.45333492,
        426.05872695, 424.5828683 , 422.28579277, 419.45340217,
        417.48451452, 413.89217512, 411.89280618, 407.87423578,
        405.66988602, 401.43096804, 398.94221266, 394.61710972,
        391.8191878 , 387.49647861, 384.39784751, 380.13497498,
        376.76469765, 372.59579444, 368.99564055, 364.93772054,
        361.15611663, 357.21318823, 353.30121144, 349.46801097,
        345.47589437, 341.74108317, 337.71570134, 334.06484383,
        330.0481559 , 326.46524576, 322.49300413, 318.96340759,
        315.06372262, 311.57538766, 307.77000414, 304.3136551 ,
        300.61673092, 297.18736492, 293.6062674 , 290.20289202,
        286.73958055, 283.36484817, 280.01648251, 276.67589128,
        273.43605502, 270.13779791, 266.99765959, 263.75063595,
        260.69982958, 257.51417795, 254.5425277 , 251.42692408,
        248.52422505, 245.48766546, 242.6460957 , 239.69400246,
        236.90604716, 234.04314711, 231.30499241, 228.53221054,
        225.84132692, 223.15836757, 220.51452564, 217.91847061,
        215.32311709, 212.80930725, 210.26456414, 207.82778802,
        205.33734666, 202.97114596, 200.53982081, 198.23607663,
        195.86660301, 193.62108687, 191.31563123, 189.12125515,
        186.88319828, 184.73640259, 182.56540208, 180.46232533,
        178.35938496, 176.29818371, 174.26089949, 172.24123834,
        170.26680952, 168.28908288, 166.37450497, 164.43984842,
        162.58086685, 160.69086905, 158.88444271, 157.04165544])


## 3. Halo mass function & 1-halo term

In [ ]:
def halomassfuncdesp(x, a, p, A):
    return A*(2/np.pi)**0.5 * (1 + (a*x**2)**(-p)) * np.exp(-a*x**2 / 2)

def deltac(Om):
    return 3/20*(12*np.pi)**(2/3.)*(1+0.0123*np.log10(Om))
    

def HMF_Watson(x, A, alpha, beta, gamma, Om):
    sigma = deltac(Om)/x
    return A*((beta/sigma)**alpha+1)*np.exp(-gamma/sigma**2)

def HMF_Warren(x, A, a, b, c, Om):
    sigma = deltac(Om)/x
    return A*((sigma)**(-a)+b)*np.exp(-c/sigma**2)

def HMF_Lourdes_FoF(x, A, b, c, d):
    A*(b*x**c+1)*np.exp(-d*x**2)


In [ ]:
nu_bins = np.array([0.5554016 , 0.56279561, 0.57034737, 0.5780613 , 0.585942  ,
       0.59399419, 0.60222277, 0.6106328 , 0.61922958, 0.62801855,
       0.63700538, 0.64619589, 0.65559619, 0.66521257, 0.67505157,
       0.68511993, 0.69542467, 0.70597309, 0.71677273, 0.7278314 ,
       0.73915722, 0.75075863, 0.76264438, 0.77482353, 0.78730549,
       0.80010008, 0.8132175 , 0.82666829, 0.84046345, 0.85461442,
       0.86913311, 0.88403188, 0.89932356, 0.91502156, 0.93113984,
       0.94769286, 0.96469572, 0.98216413, 1.0001145 , 1.0185639 ,
       1.03753009, 1.05703161, 1.0770878 , 1.09771885, 1.11894576,
       1.14079044, 1.16327584, 1.18642589, 1.21026553, 1.23482082,
       1.26011902, 1.28618861, 1.31305933, 1.34076224, 1.36932987,
       1.39879629, 1.42919708, 1.46056945, 1.49295239, 1.52638675,
       1.56091527, 1.59658271, 1.63343595, 1.67152425, 1.71089921,
       1.7516149 , 1.79372802, 1.83729812, 1.88238766, 1.92906218,
       1.97739046, 2.02744483, 2.07930124, 2.13303952, 2.18874351,
       2.24650145, 2.30640614, 2.36855522, 2.43305138, 2.50000274,
       2.56952316, 2.64173254, 2.71675717, 2.79473013, 2.87579177,
       2.96009005, 3.04778104, 3.13902937, 3.23400879, 3.33290271,
       3.43590484, 3.54321978, 3.65506371, 3.77166515, 3.89326573,
       4.02012103, 4.15250145, 4.29069319, 4.43499927, 4.58574063,
       4.74325723])


In [ ]:
HMF_Marcos = np.load('data/Pk_Marcos/HMF_Planck18_N1536_L512.npy', allow_pickle=True).item()

delta_nu_first = HMF_Marcos['nu'][1]-HMF_Marcos['nu'][0]
delta_nu_rest = HMF_Marcos['nu'][1:]-HMF_Marcos['nu'][:-1]
delta_nu_1 = np.append(delta_nu_first,delta_nu_rest)
delta_nu = nu_bins[1:]-nu_bins[:-1]

missing_fnu = 1.0-np.sum(HMF_Marcos['nufnu']/HMF_Marcos['nu']*delta_nu)
fnu_Marcos = HMF_Marcos['nufnu']/HMF_Marcos['nu']
fnu_mvir_Marcos = HMF_Marcos['nufnu_mvir']/HMF_Marcos['nu']

HMF_st = halomassfuncdesp(HMF_Marcos['nu'], 0.707, 0.3, 0.21616*(np.pi/2)**0.5)
HMF_de = halomassfuncdesp(HMF_Marcos['nu'], 0.7689, 0.2536, 0.3295)
HMF_W13 = HMF_Watson(HMF_Marcos['nu'], 0.282, 2.163, 1.406, 1.210, M_Marcos.Omega_m())/HMF_Marcos['nu']
HMF_W06 = HMF_Warren(HMF_Marcos['nu'], 0.7234, 1.625, 0.2538, 1.1982, M_Marcos.Omega_m())/HMF_Marcos['nu']

Mmean = np.sum(fnu_Marcos[0:]*delta_nu[0:]*HMF_Marcos['mass'][0:])
Mmean_st = np.sum(HMF_st[0:]*delta_nu[0:]*HMF_Marcos['mass'][0:])
Mmean_de = np.sum(HMF_de[0:]*delta_nu[0:]*HMF_Marcos['mass'][0:])
imean = np.where(Mmean<HMF_Marcos['mass'][0:])
imean_st = np.where(Mmean_st<HMF_Marcos['mass'][0:])
imean_de = np.where(Mmean_de<HMF_Marcos['mass'][0:])
numean = HMF_Marcos['nu'][imean[0][0]]
numean_st = HMF_Marcos['nu'][imean_st[0][0]]
numean_de = HMF_Marcos['nu'][imean_de[0][0]]
print(Mmean,Mmean_st,Mmean_de)

plt.errorbar(HMF_Marcos['nu'], HMF_Marcos['nu']*fnu_Marcos, yerr=HMF_Marcos['nu']*fnu_Marcos/np.sqrt(HMF_Marcos['number']), label='Measurement FoF')
plt.semilogy(HMF_Marcos['nu'], HMF_Marcos['nu']*halomassfuncdesp(HMF_Marcos['nu'], 0.707, 0.3, 0.21616*(np.pi/2)**0.5), label=r'Sheth Tormen $\Delta_\mathrm{v}$')
plt.semilogy(HMF_Marcos['nu'], HMF_Watson(HMF_Marcos['nu'], 0.282, 2.163, 1.406, 1.210, M_Marcos.Omega_m()), label=r'Watson 2013 FoF')
plt.semilogy(HMF_Marcos['nu'], HMF_Warren(HMF_Marcos['nu'], 0.7234, 1.625, 0.2538, 1.1982, M_Marcos.Omega_m()), label=r'Warren 2006 FoF')

plt.legend()
plt.xlabel(r'$\nu$')
plt.ylabel(r'$\nu f(\nu)$')

plt.show()

print(HMF_Marcos.keys())
print(np.sum(fnu_Marcos*delta_nu))


In [ ]:
istart=0
theohmf = HMF_Warren(HMF_Marcos['nu'], 0.7234, 1.625, 0.2538, 1.1982, M_Marcos.Omega_m())
number_normalised = HMF_Marcos['number'][istart:]/(HMF_Marcos['nu']*fnu_Marcos)*theohmf
number_normalised[94] = np.exp(np.interp(np.log(HMF_Marcos['mass'][94]),np.log(HMF_Marcos['mass'][3::6]), np.log(number_normalised[3::6])))
number_normalised[95] = np.exp(np.interp(np.log(HMF_Marcos['mass'][95]),np.log(HMF_Marcos['mass'][1::5]), np.log(number_normalised[1::5])))
number_normalised[97] = np.exp(np.interp(np.log(HMF_Marcos['mass'][97]),np.log(HMF_Marcos['mass'][0::3]), np.log(number_normalised[0::3])))

plt.loglog(HMF_Marcos['mass'][istart:],HMF_Marcos['number'][istart:])
plt.loglog(HMF_Marcos['mass'][istart:],number_normalised)


In [ ]:
Mmean = np.sum(fnu_Marcos[0:]*delta_nu[0:]*HMF_Marcos['mass'][0:])
Mmean_st = np.sum(HMF_st[0:]*delta_nu[0:]*HMF_Marcos['mass'][0:])
Mmean_de = np.sum(HMF_de[0:]*delta_nu[0:]*HMF_Marcos['mass'][0:])
imean = np.where(Mmean<HMF_Marcos['mass'][0:])
imean_st = np.where(Mmean_st<HMF_Marcos['mass'][0:])
imean_de = np.where(Mmean_de<HMF_Marcos['mass'][0:])
numean = HMF_Marcos['nu'][imean[0][0]]
numean_st = HMF_Marcos['nu'][imean_st[0][0]]
numean_de = HMF_Marcos['nu'][imean_de[0][0]]

mean_nu = np.sum(HMF_Marcos['nufnu']*delta_nu)
mean_nu_st = np.sum(HMF_st*HMF_Marcos['nu']*delta_nu)
mean_nu_de = np.sum(HMF_de*HMF_Marcos['nu']*delta_nu)
print(mean_nu)
imean = np.where(mean_nu<HMF_Marcos['nu'][0:])
imean_st = np.where(mean_nu_st<HMF_Marcos['nu'][0:])
imean_de = np.where(mean_nu_de<HMF_Marcos['nu'][0:])
Mmean_nu = HMF_Marcos['mass'][imean[0][0]]
Mmean_nu_st = HMF_Marcos['mass'][imean_st[0][0]]
Mmean_nu_de = HMF_Marcos['mass'][imean_de[0][0]]
print(Mmean_nu*1e-10,Mmean_nu_st*1e-10,Mmean_nu_de*1e-10)


In [ ]:
Mmean_nu = np.zeros(len(delta_nu))
Mmean_nu_mvir = np.zeros(len(delta_nu))
Mmean_nu_st = np.zeros(len(delta_nu))
Mmean_nu_de = np.zeros(len(delta_nu))
Mmean_nu_W13 = np.zeros(len(delta_nu))
Mmean_nu_W06 = np.zeros(len(delta_nu))

for i in range(len(delta_nu)):
    Mmean_nu[i] = np.sum(fnu_Marcos[0:i]*delta_nu[0:i]*HMF_Marcos['mass'][0:i])
    Mmean_nu_mvir[i] = np.sum(fnu_mvir_Marcos[0:i]*delta_nu[0:i]*HMF_Marcos['mass'][0:i])
    Mmean_nu_st[i] = np.sum(HMF_st[0:i]*delta_nu[0:i]*HMF_Marcos['mass'][0:i])
    Mmean_nu_de[i] = np.sum(HMF_de[0:i]*delta_nu[0:i]*HMF_Marcos['mass'][0:i])
    Mmean_nu_W13[i] = np.sum(HMF_W13[0:i]*delta_nu[0:i]*HMF_Marcos['mass'][0:i])
    Mmean_nu_W06[i] = np.sum(HMF_W06[0:i]*delta_nu[0:i]*HMF_Marcos['mass'][0:i])


In [ ]:
# esthetic definitions for the plots

scaling_factor = 1.0
values = fnu_Marcos*HMF_Marcos['mass']*delta_nu
theo_values = HMF_W06*HMF_Marcos['mass']*delta_nu
non_zero_values = np.where(values == 0, 1e-10, values)
values_correct = values.copy()
values_correct[94] = theo_values[94]
values_correct[95] = theo_values[95]
values_correct[97] = theo_values[97]
errors = values_correct / np.sqrt(number_normalised) / scaling_factor
fig, ax = plt.subplots(3,sharex=True,height_ratios=[2,2,1],figsize=(6.4, 8))
fig.subplots_adjust(hspace=0.1)
print(np.where(values==0))
# Replace zero counts with a small value for plotting

# Cumulative error
cumulative_variance = np.cumsum(errors**2)
cumulative_error = np.sqrt(cumulative_variance)

ax[0].errorbar(HMF_Marcos['nu'], non_zero_values, yerr=errors, fmt='_', c='k', alpha=0.75, label='Measurement FoF')
ax[0].semilogy(HMF_Marcos['nu'], halomassfuncdesp(HMF_Marcos['nu'], 0.707, 0.3, 0.21616*(np.pi/2)**0.5)*HMF_Marcos['mass']*delta_nu, label=r'Sheth Tormen $\Delta_\mathrm{v}$',c='r')
ax[0].semilogy(HMF_Marcos['nu'], HMF_W06*HMF_Marcos['mass']*delta_nu, label='Warren 2006 FoF',c='purple',ls='--')

ax[0].set_ylim([148982983.34161836,4290550615639.2915])

for i, v in enumerate(values):
    if v == 0:
        ax[0].annotate('', 
                       xy=(HMF_Marcos['nu'][i] - 0.05, ax[0].get_ylim()[0] * 1.5), 
                       xytext=(HMF_Marcos['nu'][i] - 0.05, ax[0].get_ylim()[0] * 10),
                       arrowprops=dict(facecolor='blue', edgecolor='blue', arrowstyle='-|>'))

ax[1].errorbar(HMF_Marcos['nu'], Mmean_nu, yerr=cumulative_error, label=r'$f_\mathrm{h}(\nu)$ from simulation (FoF)',fmt='_',c='k',alpha=0.75)
ax[1].semilogy(HMF_Marcos['nu'], Mmean_nu_st, label=r'$f_\mathrm{h}(\nu)$ from Sheth & Tormen 2001 ($\Delta_\mathrm{v}$)',c='r')
ax[1].semilogy(HMF_Marcos['nu'], Mmean_nu_W06, label=r'$f_\mathrm{h}(\nu)$ from Warren et al. 2006 (FoF)',c='purple',ls='--')

ax[1].legend(loc='lower right')
ax[2].errorbar(HMF_Marcos['nu'], (Mmean_nu-Mmean_nu_st)/Mmean_nu_st, yerr=cumulative_error/Mmean_nu_st, label='Measurement FoF',fmt='_',c='k',alpha=0.75)
ax[2].plot(HMF_Marcos['nu'], (Mmean_nu_st-Mmean_nu_st)/Mmean_nu_st, label=r'Sheth \& Tormen $\Delta_\mathrm{v}$',c='r')
ax[2].plot(HMF_Marcos['nu'], (Mmean_nu_W06-Mmean_nu_st)/Mmean_nu_st, label=r'Warren et al. FoF',c='purple',ls='--')

ax[2].set_xlabel(r'$\nu$')
ax[0].set_ylabel(r'$M(\nu)f_\mathrm{h}(\nu)\,\Delta\nu$')
ax[1].set_ylabel(r'$P^\mathrm{1h}(k\rightarrow 0,\nu) = \int_{\nu_0}^{\nu} M(\nu^\prime)f_\mathrm{h}(\nu^\prime)\,d\nu^\prime$')
ax[2].set_ylabel(r'$\frac{P^\mathrm{1h}(\nu) - P_\mathrm{ST}^\mathrm{1h}(\nu)}{P_\mathrm{ST}^\mathrm{1h}(\nu)}$',fontsize=14)

ax[1].set_ylim([4e11,5e13])
ax[2].set_ylim([-0.1,0.5])
print(cumulative_error/Mmean_nu)
fig.savefig(os.path.join(FIG_DIR, 'HMF_cumulative_integrand_comparison'),bbox_inches='tight')
plt.show()


In [ ]:
uKM_Marcos = np.load('data/Pk_Marcos/u_kM_Planck18_N1536_L512.npy', allow_pickle=True).item()
np.nan_to_num(uKM_Marcos['u_kM'],nan=1.0)
np.nan_to_num(uKM_Marcos['u_kM_NFW'],nan=1.0)
print(uKM_Marcos.keys())
for i in range(50):
    l1, = plt.semilogx(uKM_Marcos['k_values'], uKM_Marcos['u_kM'][i]/uKM_Marcos['u_kM'][i][0])
    c1 = l1.get_color()
    plt.semilogx(uKM_Marcos['k_values'], uKM_Marcos['u_kM_NFW'][i]/uKM_Marcos['u_kM_NFW'][i][0],ls=':',c=c1)
plt.show()


In [ ]:
import numpy as np
from scipy.special import sici

def window_filament_2f(k, d, R):
    pi = np.pi

    def window_tophat(k, R, Deltav):
        return (3 * Deltav) / (k * R)**3 * (
            np.sin(k * R / (Deltav**(1.0/3.0))) -
            (k * R / (Deltav**(1.0/3.0))) * np.cos(k * R / (Deltav**(1.0/3.0)))
        )

    si, _ = sici(2 * k * R)

    tophat_value = window_tophat(k, np.sqrt(5.0 / 6.0) * d, 1.0)

    result = 2 * (-1 + np.cos(2 * k * R) + 2 * k * R * si) / (2 * k * R)**2 * abs(tophat_value)**2

    return result


In [ ]:
Radius = (3*HMF_Marcos['mass']/(4*np.pi*86193413413.5554))**(1./3.)
thickness = Radius/(303.46432**(1./3.))
plt.loglog(HMF_Marcos['mass'],Radius)
plt.loglog(HMF_Marcos['mass'],thickness)


In [ ]:
P1h_analytical_Marcos = np.zeros(len(uKM_Marcos['k_values']))
P1h_analytical_Marcos_mvir = np.zeros(len(uKM_Marcos['k_values']))
P1h_analytical_Marcos_st = np.zeros(len(uKM_Marcos['k_values']))
P1h_analytical_Marcos_de = np.zeros(len(uKM_Marcos['k_values']))
P1h_analytical_Marcos_W13 = np.zeros(len(uKM_Marcos['k_values']))
P1h_analytical_Marcos_W06 = np.zeros(len(uKM_Marcos['k_values']))

P1h_analytical_Marcos_nonan = np.zeros(len(uKM_Marcos['k_values']))
P1h_f_analytical_Marcos = np.zeros(len(uKM_Marcos['k_values']))

print(Mmean*1e-13, Mmean_st*1e-13, Mmean_de*1e-13)

for i in range(len(P1h_analytical_Marcos)):
    P1h_analytical_Marcos[i] = 1./(86193413413.5554)*np.sum((uKM_Marcos['u_kM_NFW'][0:,i]/uKM_Marcos['u_kM_NFW'][0:,0])**2*fnu_Marcos[0:]*delta_nu[0:]*HMF_Marcos['mass'][0:])
    P1h_analytical_Marcos_mvir[i] = 1./(86193413413.5554)*np.sum((uKM_Marcos['u_kM_NFW'][0:,i]/uKM_Marcos['u_kM_NFW'][0:,0])**2*fnu_mvir_Marcos[0:]*delta_nu[0:]*HMF_Marcos['mass'][0:])
    P1h_analytical_Marcos_st[i] = 1./(86193413413.5554)*np.sum((uKM_Marcos['u_kM_NFW'][0:,i]/uKM_Marcos['u_kM_NFW'][0:,0])**2*HMF_st[0:]*delta_nu[0:]*HMF_Marcos['mass'][0:])
    P1h_analytical_Marcos_de[i] = 1./(86193413413.5554)*np.sum((uKM_Marcos['u_kM_NFW'][0:,i]/uKM_Marcos['u_kM_NFW'][0:,0])**2*HMF_de[0:]*delta_nu[0:]*HMF_Marcos['mass'][0:])
    P1h_analytical_Marcos_W13[i] = 1./(86193413413.5554)*np.sum((uKM_Marcos['u_kM_NFW'][0:,i]/uKM_Marcos['u_kM_NFW'][0:,0])**2*HMF_W13[0:]*delta_nu[0:]*HMF_Marcos['mass'][0:])
    P1h_analytical_Marcos_W06[i] = 1./(86193413413.5554)*np.sum((uKM_Marcos['u_kM_NFW'][0:,i]/uKM_Marcos['u_kM_NFW'][0:,0])**2*HMF_W06[0:]*delta_nu[0:]*HMF_Marcos['mass'][0:])
    P1h_analytical_Marcos_nonan[i] = 1./(86193413413.5554)*np.sum((uKM_Marcos['u_kM_NFW'][30:,i]/uKM_Marcos['u_kM_NFW'][30:,0])**2*fnu_Marcos[30:]*delta_nu[30:]*HMF_Marcos['mass'][30:])
    P1h_f_analytical_Marcos[i] = 1./(86193413413.5554)*np.sum(window_filament_2f(uKM_Marcos['k_values'][i], thickness, Radius)*fnu_Marcos[:]*delta_nu[:]*HMF_Marcos['mass'])
    


In [ ]:
Pk1h_Marcos_int = np.interp(uKM_Marcos['k_values'],kvec_Marcos,Pk1h_Marcos)
P1h_analytical_Marcos_int = np.interp(kvec_Marcos,uKM_Marcos['k_values'],P1h_analytical_Marcos)
P1h_f_analytical_Marcos_int = np.interp(kvec_Marcos,uKM_Marcos['k_values'],P1h_f_analytical_Marcos)
plt.semilogx(kvec_Marcos,Pk1h_Marcos/Pk1h_Marcos,label=r'measured $P_{1h}(k)$')
plt.semilogx(kvec_Marcos,P1h_analytical_Marcos_int/Pk1h_Marcos,label='constructed $P_{1h}(k)$ from measured n(M) and u(k,M)')
plt.legend()
plt.xlabel(r'$k$')
plt.ylabel(r'$P_{1H}(k)/P_{1H}^\mathrm{meas}(k)$')


## 4. Emulators (BACCO / EE2)

In [ ]:
emulator = baccoemu.Matter_powerspectrum()
if SAMPLE == 'BACCO':
    for i in range(0,Ncosmo):
        pk_lin_total[i] = {}
        pk_nl_total[i] = {}
        for j in range(0,3):
            print(i,j)
            print('(sigma8, sigma8_cb, Mnu)=(%.3f, %.3f, %.3f) at i=%d'%(s8[i], s8_cb[i], Mnu[i], i))
            params = {
                'omega_cold'    :  Om_cb[i],
                'A_s'           :  A_s[i], # if sigma8_cold is not specified
                #'sigma8_cold'   :  s8_cb[i], # if sigma8_cold is not specified
                'omega_baryon'  :  Om_b[i],
                'ns'            :  n_s[i],
                'hubble'        :  hubble[i],
                'neutrino_mass' :  Mnu[i],
                'w0'            :  w0[i],
                'wa'            :  wa[i],
                'expfactor'     :  avec[j]
            }
            kvecbacco, pk_lin_total[i][j] = emulator.get_linear_pk(k=kvecbacco, cold=False, **params)
            kvecbacco, pk_nl_total[i][j] = emulator.get_nonlinear_pk(k=kvecbacco, cold=False, **params)
pk_nl_fid = {}
pk_lin_fid = {}
for j in range(0,3):
    params_fid = {
        'omega_cold'    :  (om_cdm_fid+om_b_fid)/hubble_fid**2,
        'A_s'           :  A_s_fid, # if sigma8_cold is not specified
        'omega_baryon'  :  om_b_fid/hubble_fid**2,
        'ns'            :  n_s_fid,
        'hubble'        :  hubble_fid,
        'neutrino_mass' :  0.06,
        'w0'            : -1.0,
        'wa'            :  0.0,
        'expfactor'     :  avec[j]
    }
    kvecbacco, pk_lin_fid[j] = emulator.get_linear_pk(k=kvecbacco, cold=False, **params_fid)
    kvecbacco, pk_nl_fid[j] = emulator.get_nonlinear_pk(k=kvecbacco, cold=False, **params_fid)

for i in range(0,3):
    pk_lin_bacco[i] = {}
    pk_nl_bacco[i] = {}
    for j in range(0,3):
        params_bacco = {
            'omega_cold'    :  (om_cdm_bacco[i]+om_b_bacco[i])/hubble_bacco[i]**2,
            'sigma8_cold'   :  s8_bacco[i], # if sigma8_cold is not specified
            'omega_baryon'  :  om_b_bacco[i]/hubble_bacco[i]**2,
            'ns'            :  n_s_bacco[i],
            'hubble'        :  hubble_bacco[i],
            'neutrino_mass' :  0.0,
            'w0'            : -1.0,
            'wa'            :  0.0,
            'expfactor'     :  avec[j]
        }
        kvecbacco, pk_lin_bacco[i][j] = emulator.get_linear_pk(k=kvecbacco, cold=False, **params_bacco)
        kvecbacco, pk_nl_bacco[i][j] = emulator.get_nonlinear_pk(k=kvecbacco, cold=False, **params_bacco)


In [ ]:
if SAMPLE == 'EE2':
    pk_nl_ee2 = {}
    pk_lin_ee2 = {}
    for i in range(0,Ncosmo):
        pk_nl_ee2[i] = {}
        pk_lin_ee2[i] = {}
        params = {
            'Omm'    :  M[i].get_current_derived_parameters(['Omega_m'])['Omega_m'],
            'As'           :  A_s[i],
            'Omb'  :  Om_b[i],
            'ns'            :  n_s[i],
            'h'        :  hubble[i],
            'mnu' :  Mnu[i],
            'w'            : w0[i],
            'wa'            :  wa[i],
        }
        k, pk_nl_ee2[i], pk_lin_ee2[i], b = ee2.get_pnonlin(params,redshifts,kvec_sample)


In [ ]:
# Reference non-linear spectra of the selected sample on kvec_sample, cached under data/ (used by the Fig. 5 cell;
# the LPT-scatter notebook reads the same files): BACCO -> baccoemu, EE2 -> EuclidEmulator2, CosmicEmu -> the Mira-Titan spectra.
from scipy.interpolate import InterpolatedUnivariateSpline
pk_nl_ref = {}
for i in range(0,Ncosmo):
    if SAMPLE == 'BACCO':
        _pk = load_or_compute('data/PkBacco/PkBacco_w0wamnu_%d.txt'%(i),
                              lambda: np.c_[kvec_sample, np.interp(kvec_sample, kvecbacco, pk_nl_total[i][0]), np.interp(kvec_sample, kvecbacco, pk_nl_total[i][1]), np.interp(kvec_sample, kvecbacco, pk_nl_total[i][2])],
                              recompute=RECOMPUTE['bacco'], write=WRITE['bacco'])
    elif SAMPLE == 'EE2':
        _pk = load_or_compute('data/PkEE2/PkEE2_w0wamnu_%d.txt'%(i),
                              lambda: np.c_[kvec_sample, pk_nl_ee2[i][0], pk_nl_ee2[i][1], pk_nl_ee2[i][2]],
                              recompute=RECOMPUTE['ee2'], write=WRITE['ee2'])
    else:
        _cols = [kvec_sample]
        for j, folder in enumerate(['z0','z1','z2']):
            _k, _p = np.loadtxt('data/CosmicEmu/%s/EMU%d.txt'%(folder,i), unpack=True)
            _cols.append(np.exp(InterpolatedUnivariateSpline(np.log(_k), np.log(_p), k=1)(np.log(kvec_sample))))
        _pk = np.c_[tuple(_cols)]
    pk_nl_ref[i] = {iz: _pk[:,iz+1] for iz in range(len(redshifts))}


## 5. CAMB & HMcode runs

In [ ]:
W = dict(HMCode_A_baryon=4.0, HMCode_eta_baryon=0.0)
WT = dict(W, WHM_thinweb=1)
whmcode_cases = [(9,{}), (6,{}), (16,{}), (17,W), (23,W), (24,W), (25,W), (14,W), (10,dict(HMCode_logT_AGN=5.0)), (10,dict(HMCode_logT_AGN=10.0)),
                 (23,WT), (24,WT), (25,WT), (14,WT)]  # last four: 'thin web' sheet/filament profiles (upper edges of the bands in Fig. 4)
cp_fid = {}
CambRun_fid = {}
for j in range(0,len(whmcode_cases)):
    cp_fid[j] = camb.CAMBparams()
    cp_fid[j].set_cosmology(H0=hubble_fid*100,
                     omch2=om_cdm_fid,
                     ombh2=om_b_fid,
                     YHe=YHe,
                     TCMB=T_CMB,
                    )
    cp_fid[j].InitPower.set_params(ns=n_s_fid, As=A_s_fid)
    cp_fid[j].MassiveNuMethod='Nu_best'
    cp_fid[j].omnuh2=0.06/93.14 
    cp_fid[j].num_nu_massive=1
    cp_fid[j].num_nu_massless=0.00441
    cp_fid[j].nu_mass_degeneracies=[3.03959]
    cp_fid[j].scalar_initial_condition=1
    cp_fid[j].WantTransfer = True
    cp_fid[j].WantCls = False
    cp_fid[j].Want_CMB = False
    cp_fid[j].Want_CMB_lensing = False
    cp_fid[j].DoLensing = False
    cp_fid[j].AccurateReionization = False

    cp_fid[j].NonLinear = camb.model.NonLinear_both
    cp_fid[j].NonLinearModel.set_params(halofit_version=whmcode_cases[j][0], **whmcode_cases[j][1])
    cp_fid[j].set_matter_power(redshifts=np.linspace(0,10,num=100),
                        kmax=kmax_for_computation,
                        accurate_massive_neutrino_transfers=True)

    if (want_output_feedback):
        camb.set_feedback_level(level=3)
    else:
        camb.set_feedback_level(level=0)

    if (high_precision):
        cp_fid[j].set_accuracy(
                        DoLateRadTruncation=False,
                        AccuracyBoost=5,
                        lAccuracyBoost=5,
                        lSampleBoost=5,
                        )
    else:
        cp_fid[j].set_accuracy(
                        DoLateRadTruncation=False,
                        AccuracyBoost=1,
                        lAccuracyBoost=1,
                        )    
    start = time.time()
    CambRun_fid[j] = camb.get_results(cp_fid[j])
    print("CAMB run took %f seconds"%(time.time()-start))


In [ ]:
Pkl_hmc_fid = np.zeros((len(redshifts),len(kvec))) 
Pk_hmc_fid = np.zeros((len(redshifts),len(kvec))) 
Pk_ori_fid = np.zeros((len(redshifts),len(kvec)))
Pk_ori_tweaked_fid = np.zeros((len(redshifts),len(kvec)))
Pk_whmc_halo = np.zeros((len(redshifts),len(kvec)))
Pk_whmc_halominussphere = np.zeros((len(redshifts),len(kvec)))
Pk_whmc_filaminussphere = np.zeros((len(redshifts),len(kvec)))
Pk_whmc_sheetminussphere = np.zeros((len(redshifts),len(kvec)))
Pk_whmc_full_tweaked = np.zeros((len(redshifts),len(kvec)))
Pk_hmc_bf_low = np.zeros((len(redshifts),len(kvec)))
Pk_hmc_bf_high = np.zeros((len(redshifts),len(kvec)))

PKL_hmc_fid = CambRun_fid[0].get_matter_power_interpolator(nonlinear=False, hubble_units=True, k_hunit=True)
PKNL_hmc_fid = CambRun_fid[0].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_ori_fid = CambRun_fid[1].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_ori_tweaked_fid = CambRun_fid[2].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_whmc_halo = CambRun_fid[3].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_whmc_halominussphere = CambRun_fid[4].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_whmc_filaminussphere = CambRun_fid[5].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_whmc_sheetminussphere = CambRun_fid[6].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_whmc_full_tweaked = CambRun_fid[7].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_hmc_bf_low= CambRun_fid[8].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_hmc_bf_high= CambRun_fid[9].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)

for iz in range(0,len(redshifts)):
    for ik in range(0,len(kvec)):
        Pkl_hmc_fid[iz][ik] = PKL_hmc_fid.P(redshifts[iz],kvec[ik])
        Pk_hmc_fid[iz][ik] = PKNL_hmc_fid.P(redshifts[iz],kvec[ik])
        Pk_ori_fid[iz][ik] = PKNL_ori_fid.P(redshifts[iz],kvec[ik])
        Pk_ori_tweaked_fid[iz][ik] = PKNL_ori_tweaked_fid.P(redshifts[iz],kvec[ik])
        Pk_whmc_halo[iz][ik] = PKNL_whmc_halo.P(redshifts[iz],kvec[ik])
        Pk_whmc_halominussphere[iz][ik] = PKNL_whmc_halominussphere.P(redshifts[iz],kvec[ik])
        Pk_whmc_filaminussphere[iz][ik] = PKNL_whmc_filaminussphere.P(redshifts[iz],kvec[ik])
        Pk_whmc_sheetminussphere[iz][ik] = PKNL_whmc_sheetminussphere.P(redshifts[iz],kvec[ik])
        Pk_whmc_full_tweaked[iz][ik] = PKNL_whmc_full_tweaked.P(redshifts[iz],kvec[ik])
        Pk_hmc_bf_low[iz][ik] = PKNL_hmc_bf_low.P(redshifts[iz],kvec[ik])
        Pk_hmc_bf_high[iz][ik] = PKNL_hmc_bf_high.P(redshifts[iz],kvec[ik])

# 'thin web' variants (cases 10-13 of whmcode_cases)
Pk_whmc_halominussphere_concnfw = np.zeros((len(redshifts),len(kvec)))
Pk_whmc_filaminussphere_concnfw = np.zeros((len(redshifts),len(kvec)))
Pk_whmc_sheetminussphere_concnfw = np.zeros((len(redshifts),len(kvec)))
Pk_whmc_full_tweaked_concnfw = np.zeros((len(redshifts),len(kvec)))
_tw = {name: CambRun_fid[j].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
       for name, j in [('h',10), ('f',11), ('s',12), ('t',13)]}
for iz in range(0,len(redshifts)):
    for ik in range(0,len(kvec)):
        Pk_whmc_halominussphere_concnfw[iz][ik] = _tw['h'].P(redshifts[iz],kvec[ik])
        Pk_whmc_filaminussphere_concnfw[iz][ik] = _tw['f'].P(redshifts[iz],kvec[ik])
        Pk_whmc_sheetminussphere_concnfw[iz][ik] = _tw['s'].P(redshifts[iz],kvec[ik])
        Pk_whmc_full_tweaked_concnfw[iz][ik] = _tw['t'].P(redshifts[iz],kvec[ik])


In [ ]:
# deal with the CAMB runs
cp_bacco = {}
CambRun_bacco = {}
halofit = [(9,{}), (6,{}), (16,{}), (17,W), (23,W), (24,W), (25,W), (14,W), (10,dict(HMCode_logT_AGN=5.0)), (10,dict(HMCode_logT_AGN=8.3)),
                 (23,WT), (24,WT), (25,WT), (14,WT)]  # last four: 'thin web' sheet/filament profiles (upper edges of the bands in Fig. 4)
for i in range(0,3):
    cp_bacco[i] = {}
    CambRun_bacco[i] = {}
    for j in range(0,len(halofit)):
        cp_bacco[i][j] = camb.CAMBparams()
        cp_bacco[i][j].set_cosmology(H0=hubble_bacco[i]*100,
                         omch2=om_cdm_bacco[i],
                         ombh2=om_b_bacco[i],
                         YHe=YHe,
                         TCMB=T_CMB,
                        )
        cp_bacco[i][j].InitPower.set_params(ns=n_s_bacco[i], As=M_bacco[i].get_current_derived_parameters(['A_s'])['A_s'])#*s8_corr[i])
        cp_bacco[i][j].scalar_initial_condition=1
        cp_bacco[i][j].num_nu_massive=0
        cp_bacco[i][j].num_nu_massless=3.044
        cp_bacco[i][j].nu_mass_numbers=[0]
        cp_bacco[i][j].WantTransfer = True
        cp_bacco[i][j].WantCls = False
        cp_bacco[i][j].Want_CMB = False
        cp_bacco[i][j].Want_CMB_lensing = False
        cp_bacco[i][j].DoLensing = False
        cp_bacco[i][j].AccurateReionization = False

        cp_bacco[i][j].NonLinear = camb.model.NonLinear_both
        cp_bacco[i][j].NonLinearModel.set_params(halofit_version=halofit[j][0], **halofit[j][1])
        cp_bacco[i][j].set_matter_power(redshifts=np.linspace(0,10,num=100),
                            kmax=kmax_for_computation,
                            accurate_massive_neutrino_transfers=False)

        if (want_output_feedback):
            camb.set_feedback_level(level=3)
        else:
            camb.set_feedback_level(level=0)

        if (high_precision):
            cp_bacco[i][j].set_accuracy(
                            DoLateRadTruncation=False,
                            AccuracyBoost=5,
                            lAccuracyBoost=5,
                            lSampleBoost=5,
                            )
        else:
            cp_bacco[i][j].set_accuracy(
                            DoLateRadTruncation=False,
                            AccuracyBoost=1,
                            lAccuracyBoost=1,
                            )    
        start = time.time()
        CambRun_bacco[i][j] = camb.get_results(cp_bacco[i][j])
        print("CAMB run %d-%d took %f seconds"%(i, j, time.time()-start))


In [ ]:
Pkl_hmc_bacco = np.zeros((3,len(redshifts),len(kvec))) 
Pk_hmc_bacco = np.zeros((3,len(redshifts),len(kvec))) 
Pk_ori_bacco = np.zeros((3,len(redshifts),len(kvec)))
Pk_ori_tweaked_bacco = np.zeros((3,len(redshifts),len(kvec)))
Pk_whmc_halo_bacco = np.zeros((3,len(redshifts),len(kvec)))
Pk_whmc_halominussphere_bacco = np.zeros((3,len(redshifts),len(kvec)))
Pk_whmc_filaminussphere_bacco = np.zeros((3,len(redshifts),len(kvec)))
Pk_whmc_sheetminussphere_bacco = np.zeros((3,len(redshifts),len(kvec)))
Pk_whmc_full_tweaked_bacco = np.zeros((3,len(redshifts),len(kvec)))
Pk_hmc_bf_low_bacco = np.zeros((3,len(redshifts),len(kvec)))
Pk_hmc_bf_high_bacco = np.zeros((3,len(redshifts),len(kvec)))
for i in range(3):
    PKL_hmc_bacco = CambRun_bacco[i][0].get_matter_power_interpolator(nonlinear=False, hubble_units=True, k_hunit=True)
    PKNL_hmc_bacco = CambRun_bacco[i][0].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
    PKNL_ori_bacco = CambRun_bacco[i][1].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
    PKNL_ori_tweaked_bacco = CambRun_bacco[i][2].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
    PKNL_whmc_halo_bacco = CambRun_bacco[i][3].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
    PKNL_whmc_halominussphere_bacco = CambRun_bacco[i][4].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
    PKNL_whmc_filaminussphere_bacco = CambRun_bacco[i][5].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
    PKNL_whmc_sheetminussphere_bacco = CambRun_bacco[i][6].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
    PKNL_whmc_full_tweaked_bacco = CambRun_bacco[i][7].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
    PKNL_hmc_bf_low_bacco= CambRun_bacco[i][8].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
    PKNL_hmc_bf_high_bacco= CambRun_bacco[i][9].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
    
    for iz in range(0,len(redshifts)):
        for ik in range(0,len(kvec)):
            Pkl_hmc_bacco[i][iz][ik] = PKL_hmc_bacco.P(redshifts[iz],kvec[ik])
            Pk_hmc_bacco[i][iz][ik] = PKNL_hmc_bacco.P(redshifts[iz],kvec[ik])
            Pk_ori_bacco[i][iz][ik] = PKNL_ori_bacco.P(redshifts[iz],kvec[ik])
            Pk_ori_tweaked_bacco[i][iz][ik] = PKNL_ori_tweaked_bacco.P(redshifts[iz],kvec[ik])
            Pk_whmc_halo_bacco[i][iz][ik] = PKNL_whmc_halo_bacco.P(redshifts[iz],kvec[ik])
            Pk_whmc_halominussphere_bacco[i][iz][ik] = PKNL_whmc_halominussphere_bacco.P(redshifts[iz],kvec[ik])
            Pk_whmc_filaminussphere_bacco[i][iz][ik] = PKNL_whmc_filaminussphere_bacco.P(redshifts[iz],kvec[ik])
            Pk_whmc_sheetminussphere_bacco[i][iz][ik] = PKNL_whmc_sheetminussphere_bacco.P(redshifts[iz],kvec[ik])
            Pk_whmc_full_tweaked_bacco[i][iz][ik] = PKNL_whmc_full_tweaked_bacco.P(redshifts[iz],kvec[ik])
            Pk_hmc_bf_low_bacco[i][iz][ik] = PKNL_hmc_bf_low_bacco.P(redshifts[iz],kvec[ik])
            Pk_hmc_bf_high_bacco[i][iz][ik] = PKNL_hmc_bf_high_bacco.P(redshifts[iz],kvec[ik])

# 'thin web' variants (cases 10-13 of halofit)
Pk_whmc_halominussphere_bacco_concnfw = np.zeros((3,len(redshifts),len(kvec)))
Pk_whmc_filaminussphere_bacco_concnfw = np.zeros((3,len(redshifts),len(kvec)))
Pk_whmc_sheetminussphere_bacco_concnfw = np.zeros((3,len(redshifts),len(kvec)))
Pk_whmc_full_tweaked_bacco_concnfw = np.zeros((3,len(redshifts),len(kvec)))
for i in range(3):
    _tw = {name: CambRun_bacco[i][j].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
           for name, j in [('h',10), ('f',11), ('s',12), ('t',13)]}
    for iz in range(0,len(redshifts)):
        for ik in range(0,len(kvec)):
            Pk_whmc_halominussphere_bacco_concnfw[i][iz][ik] = _tw['h'].P(redshifts[iz],kvec[ik])
            Pk_whmc_filaminussphere_bacco_concnfw[i][iz][ik] = _tw['f'].P(redshifts[iz],kvec[ik])
            Pk_whmc_sheetminussphere_bacco_concnfw[i][iz][ik] = _tw['s'].P(redshifts[iz],kvec[ik])
            Pk_whmc_full_tweaked_bacco_concnfw[i][iz][ik] = _tw['t'].P(redshifts[iz],kvec[ik])


In [ ]:
# deal with the CAMB runs
# Marcos cosmology: WHM modes with CAMB's default HMcode baryon parameters; WW = Warren et al. (2006) halo mass function (purple curves of Fig. 8)
WW = dict(WHM_hmf=1)
halofit = [(9,{}), (6,{}), (16,{}), (17,{}), (20,{}), (23,{}), (24,{}), (25,{}), (14,{}), (10,dict(HMCode_logT_AGN=5.0)), (10,dict(HMCode_logT_AGN=10.0)),
           (17,WW), (23,WW), (14,WW)]
cp_Marcos = {}
CambRun_Marcos = {}
for j in range(0,len(halofit)):
    cp_Marcos[j] = camb.CAMBparams()
    cp_Marcos[j].set_cosmology(H0=0.6766*100,
                     omch2=0.26067*0.6766*0.6766,
                     ombh2=0.04897*0.6766*0.6766,
                     YHe=YHe,
                     TCMB=T_CMB,
                    )
    cp_Marcos[j].InitPower.set_params(ns=0.9665, As=M_Marcos.get_current_derived_parameters(['A_s'])['A_s'])#*s8_corr[i])
    cp_Marcos[j].scalar_initial_condition=1
    cp_Marcos[j].num_nu_massive=0
    cp_Marcos[j].num_nu_massless=3.046
    cp_Marcos[j].nu_mass_numbers=[0]
    cp_Marcos[j].WantTransfer = True
    cp_Marcos[j].WantCls = False
    cp_Marcos[j].Want_CMB = False
    cp_Marcos[j].Want_CMB_lensing = False
    cp_Marcos[j].DoLensing = False
    cp_Marcos[j].AccurateReionization = False

    cp_Marcos[j].NonLinear = camb.model.NonLinear_both
    cp_Marcos[j].NonLinearModel.set_params(halofit_version=halofit[j][0], **halofit[j][1])
    cp_Marcos[j].set_matter_power(redshifts=np.linspace(0,10,num=100),
                        kmax=kmax_for_computation,
                        accurate_massive_neutrino_transfers=False)

    if (want_output_feedback):
        camb.set_feedback_level(level=3)
    else:
        camb.set_feedback_level(level=0)

    if (high_precision):
        cp_Marcos[j].set_accuracy(
                        DoLateRadTruncation=False,
                        AccuracyBoost=5,
                        lAccuracyBoost=5,
                        lSampleBoost=5,
                        )
    else:
        cp_Marcos[j].set_accuracy(
                        DoLateRadTruncation=False,
                        AccuracyBoost=1,
                        lAccuracyBoost=1,
                        )    
    start = time.time()
    CambRun_Marcos[j] = camb.get_results(cp_Marcos[j])
    print("CAMB run %d-%d took %f seconds"%(i, j, time.time()-start))


In [ ]:
Pkl_hmc_Marcos_long = np.zeros((len(redshifts),len(kvec))) 
Pkl_hmc_Marcos = np.zeros((len(redshifts),len(kvec_Marcos))) 
Pk_hmc_Marcos = np.zeros((len(redshifts),len(kvec_Marcos))) 
Pk_ori_Marcos = np.zeros((len(redshifts),len(kvec_Marcos)))
Pk_ori_tweaked_Marcos = np.zeros((len(redshifts),len(kvec_Marcos)))
Pk_whmc_halo_Marcos = np.zeros((len(redshifts),len(kvec_Marcos)))
Pk_whmc_halosphere_Marcos = np.zeros((len(redshifts),len(kvec_Marcos)))
Pk_whmc_halominussphere_Marcos = np.zeros((len(redshifts),len(kvec_Marcos)))
Pk_whmc_filaminussphere_Marcos = np.zeros((len(redshifts),len(kvec_Marcos)))
Pk_whmc_sheetminussphere_Marcos = np.zeros((len(redshifts),len(kvec_Marcos)))
Pk_whmc_full_tweaked_Marcos = np.zeros((len(redshifts),len(kvec_Marcos)))
Pk_hmc_bf_low_Marcos = np.zeros((len(redshifts),len(kvec_Marcos)))
Pk_hmc_bf_high_Marcos = np.zeros((len(redshifts),len(kvec_Marcos)))

PKL_hmc_Marcos = CambRun_Marcos[0].get_matter_power_interpolator(nonlinear=False, hubble_units=True, k_hunit=True)
PKNL_hmc_Marcos = CambRun_Marcos[0].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_ori_Marcos = CambRun_Marcos[1].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_ori_tweaked_Marcos = CambRun_Marcos[2].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_whmc_halo_Marcos = CambRun_Marcos[3].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_whmc_halosphere_Marcos = CambRun_Marcos[4].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_whmc_halominussphere_Marcos = CambRun_Marcos[5].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_whmc_filaminussphere_Marcos = CambRun_Marcos[6].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_whmc_sheetminussphere_Marcos = CambRun_Marcos[7].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_whmc_full_tweaked_Marcos = CambRun_Marcos[8].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_hmc_bf_low_Marcos= CambRun_Marcos[9].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
PKNL_hmc_bf_high_Marcos= CambRun_Marcos[10].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)

for iz in range(0,len(redshifts)):
    for ik in range(0,len(kvec)):
        Pkl_hmc_Marcos_long[iz][ik] = PKL_hmc_Marcos.P(redshifts[iz],kvec[ik])
        Pkl_hmc_Marcos[iz][ik] = PKL_hmc_Marcos.P(redshifts[iz],kvec_Marcos[ik])
        Pk_hmc_Marcos[iz][ik] = PKNL_hmc_Marcos.P(redshifts[iz],kvec_Marcos[ik])
        Pk_ori_Marcos[iz][ik] = PKNL_ori_Marcos.P(redshifts[iz],kvec_Marcos[ik])
        Pk_ori_tweaked_Marcos[iz][ik] = PKNL_ori_tweaked_Marcos.P(redshifts[iz],kvec_Marcos[ik])
        Pk_whmc_halo_Marcos[iz][ik] = PKNL_whmc_halo_Marcos.P(redshifts[iz],kvec_Marcos[ik])
        Pk_whmc_halosphere_Marcos[iz][ik] = PKNL_whmc_halosphere_Marcos.P(redshifts[iz],kvec_Marcos[ik])
        Pk_whmc_halominussphere_Marcos[iz][ik] = PKNL_whmc_halominussphere_Marcos.P(redshifts[iz],kvec_Marcos[ik])
        Pk_whmc_filaminussphere_Marcos[iz][ik] = PKNL_whmc_filaminussphere_Marcos.P(redshifts[iz],kvec_Marcos[ik])
        Pk_whmc_sheetminussphere_Marcos[iz][ik] = PKNL_whmc_sheetminussphere_Marcos.P(redshifts[iz],kvec_Marcos[ik])
        Pk_whmc_full_tweaked_Marcos[iz][ik] = PKNL_whmc_full_tweaked_Marcos.P(redshifts[iz],kvec_Marcos[ik])
        Pk_hmc_bf_low_Marcos[iz][ik] = PKNL_hmc_bf_low_Marcos.P(redshifts[iz],kvec_Marcos[ik])
        Pk_hmc_bf_high_Marcos[iz][ik] = PKNL_hmc_bf_high_Marcos.P(redshifts[iz],kvec_Marcos[ik])

# Warren et al. (2006) halo-mass-function variants (cases 11-13 of halofit), used for the purple curves of Fig. 8
Pk_whmc_halo_Marcos_warren = np.zeros((len(redshifts),len(kvec_Marcos)))
Pk_whmc_halominussphere_Marcos_warren = np.zeros((len(redshifts),len(kvec_Marcos)))
Pk_whmc_full_tweaked_Marcos_warren = np.zeros((len(redshifts),len(kvec_Marcos)))
_ww = {name: CambRun_Marcos[j].get_matter_power_interpolator(nonlinear=True, hubble_units=True, k_hunit=True)
       for name, j in [('halo',11), ('hms',12), ('tw',13)]}
for iz in range(0,len(redshifts)):
    for ik in range(0,len(kvec_Marcos)):
        Pk_whmc_halo_Marcos_warren[iz][ik] = _ww['halo'].P(redshifts[iz],kvec_Marcos[ik])
        Pk_whmc_halominussphere_Marcos_warren[iz][ik] = _ww['hms'].P(redshifts[iz],kvec_Marcos[ik])
        Pk_whmc_full_tweaked_Marcos_warren[iz][ik] = _ww['tw'].P(redshifts[iz],kvec_Marcos[ik])


In [ ]:
knl_fid = 1.0*M_fid.nonlinear_scale_cb(redshifts, len(redshifts))
kv_fid = 1.0/M_fid.fourier_hmcode_sigmadisp_cb(redshifts, len(redshifts))
print(knl_fid)
print(kv_fid)


In [ ]:
knl_bacco = np.zeros((3,len(redshifts)))
kv_bacco = np.zeros((3,len(redshifts)))
for i in range(3):
    knl_bacco[i] = 1.0*M_bacco[i].nonlinear_scale(redshifts, len(redshifts))
    kv_bacco[i] = 1.0/M_bacco[i].fourier_hmcode_sigmadisp(redshifts, len(redshifts))
    print(knl_bacco[i])
    print(kv_bacco[i])


In [ ]:
knl_Marcos = 1.0*M_Marcos.nonlinear_scale(redshifts, len(redshifts))
kv_Marcos = 1.0/M_Marcos.fourier_hmcode_sigmadisp(redshifts, len(redshifts))
print(knl_Marcos)
print(kv_Marcos)


## 6. Velocileptors LPT (2-halo term)

In [ ]:
Pk_lpt_fid = np.zeros((len(redshifts),len(kvec)))

for iz in range(0,len(redshifts)):
    lpt = LPT_RSD(kvec*M_fid.h(),Pkl_hmc_fid[iz]/M_fid.h()**3.,
                  kIR=knl_fid[iz]*np.sqrt(5)*1.0,
                  one_loop=True,
                  cutoff=knl_fid[iz]*np.sqrt(5)*1.0, #previous default
                  jn=15, **LPT_GRID)
    
    bias_table_zeros = np.zeros(11)
    dummy, Pk_lpt_fid[iz], dummy, dummy = lpt.make_pell_fixedbias(0, bias_table_zeros, kv=kvec*M_fid.h())
    
    Pk_lpt_fid[iz]*=M_fid.h()**3. 


In [ ]:
Pk_lpt_bacco = np.zeros((3,len(redshifts),len(kvec)))
for i in range(0,3):
    for iz in range(0,len(redshifts)):
        lpt = LPT_RSD(kvec*M_bacco[i].h(),Pkl_hmc_bacco[i][iz]/M_bacco[i].h()**3.,
                      kIR=knl_bacco[i][iz]*np.sqrt(5)*1.0,
                      one_loop=True,
                      cutoff=knl_bacco[i][iz]*np.sqrt(5)*1.0, #previous default
                      jn=15, **LPT_GRID)
    
        
        bias_table_zeros = np.zeros(11)
        dummy, Pk_lpt_bacco[i][iz], dummy, dummy = lpt.make_pell_fixedbias(0, bias_table_zeros, kv=kvec*M_bacco[i].h())
        Pk_lpt_bacco[i][iz]*=M_bacco[i].h()**3. 


In [ ]:
Pk_lpt_Marcos = np.zeros((len(redshifts),len(kvec_Marcos)))

for iz in range(0,len(redshifts)):
    lpt = LPT_RSD(kvec*0.6766,Pkl_hmc_Marcos_long[iz]/0.6766**3.,
                  kIR=knl_Marcos[iz]*np.sqrt(5)*1.0,
                  one_loop=True,
                  cutoff=knl_Marcos[iz]*np.sqrt(5)*1.0, #previous default
                  jn=15, **LPT_GRID)    
    bias_table_zeros = np.zeros(11)
    dummy, Pk_lpt_Marcos[iz], dummy, dummy = lpt.make_pell_fixedbias(0, bias_table_zeros, kv=kvec_Marcos*0.6766)
    Pk_lpt_Marcos[iz]*=0.6766**3. 


In [ ]:
kIR1 = [knl_fid[0]*np.sqrt(5)]
kIR2 = [10, 5*np.sqrt(5)*knl_fid[0], 2*np.sqrt(5)*knl_fid[0], np.sqrt(5)*knl_fid[0], 0.5*np.sqrt(5)*knl_fid[0]]
cutoff1 = [knl_fid[0]*np.sqrt(5)]
cutoff2 = [10, 5*np.sqrt(5)*knl_fid[0], 2*np.sqrt(5)*knl_fid[0], np.sqrt(5)*knl_fid[0], 0.5*np.sqrt(5)*knl_fid[0]]
cutoff_jn2 = [15]
cutoff_jn1 = [5,8,12,15]

P0vec1 = {}
for i in range(0, len(kIR1)):
    P0vec1[i] = {}
    for j in range(0, len(cutoff_jn1)):
        print(j)
        P0vec1[i][j] = {}
        for r in range(0, 1):
            lpt = LPT_RSD(kvec*M_fid.h(),Pkl_hmc_fid[0]/M_fid.h()**3.,kIR=kIR1[i],one_loop=True,cutoff=cutoff1[i],jn=cutoff_jn1[j], **LPT_GRID)
            dummy, P0vec1[i][j][r], dummy, dummy = lpt.make_pell_fixedbias(0, np.zeros(11), kv=kvec*M_fid.h())#, nmax=nmax[j])
            P0vec1[i][j][r]*=M_fid.h()**3. 

P0vec2 = {}
for i in range(0, len(kIR2)):
    P0vec2[i] = {}
    for j in range(0, len(cutoff_jn2)):
        print(j)
        P0vec2[i][j] = {}
        for r in range(0, 1):
            lpt = LPT_RSD(kvec*M_fid.h(),Pkl_hmc_fid[0]/M_fid.h()**3.,kIR=kIR2[i],one_loop=True,cutoff=cutoff2[i],jn=cutoff_jn2[j], **LPT_GRID)
            dummy, P0vec2[i][j][r], dummy, dummy = lpt.make_pell_fixedbias(0, np.zeros(11), kv=kvec*M_fid.h())#, nmax=nmax[j])
            P0vec2[i][j][r]*=M_fid.h()**3.             

#from velocileptors.EPT.moment_expansion_fftw import MomentExpansion


In [ ]:
import matplotlib.pyplot as plt
import matplotlib

# Apply the third code block's formatting
font = {'size'   : 16, 'family':'STIXGeneral'}
matplotlib.rc('font', **font)
matplotlib.rc('text', usetex=True)
plt.rcParams['legend.fontsize']='x-small'
plt.rcParams['axes.linewidth'] = 2
plt.rcParams["figure.figsize"] = [5.0,6.0]

label_cutoff = ['$k_\mathrm{cutoff} = 10 h/\mathrm{Mpc}$',
                '$k_\mathrm{cutoff} = 5\sqrt{5}k_\mathrm{nl}$',
                '$k_\mathrm{cutoff} = 2\sqrt{5}k_\mathrm{nl}$',
                '$k_\mathrm{cutoff} = 1\sqrt{5}k_\mathrm{nl}$',
                '$k_\mathrm{cutoff} = \\frac{1}{2}\sqrt{5}k_\mathrm{nl}$',
               ] 

# Create subplots with shared x and y axes
fig, ax = plt.subplots(2, 1, sharex=True, sharey='row', gridspec_kw={'hspace': 0.0, 'wspace': 0.0})

# First subplot
ax[0].semilogx(kvec, Pkl_hmc_fid[0]/Pkl_hmc_fid[0], label='linear')
ax[0].semilogx(kvec, pk_nl_fid[0]/Pkl_hmc_fid[0], label='BACCO', c='k')
ls = ['--','--','--','-','--']
c = ['darkorange','darkgreen','red','magenta','violet']
for i in range(len(kIR1)):
    for j in range(len(cutoff_jn1)):
        for r in range(1):
            ax[0].semilogx(kvec, P0vec1[i][j][r]/Pkl_hmc_fid[0], ls=ls[j], c=c[j], alpha=0.8, label=f'$j = {cutoff_jn1[j]}$')

ax[0].axvline(kv_fid[0]/M_fid.h(), ls=':', c='k', label=r'$\sigma_\mathrm{v}^{-1}$')
ax[0].axvline(knl_fid[0]/M_fid.h(), ls='--', c='k', label=r'$r_\mathrm{nl}^{-1}$')
ax[0].legend(loc='upper left',bbox_to_anchor=(1.01,1.0))
ax[0].set_xlim([0.01, 4])
ax[0].set_ylim([-0.1, 2.1])
ax[0].set_ylabel(r'$\frac{P^\mathrm{X}(k)}{P^\mathrm{Lin}(k)}$', fontsize=28)

textstr = r'$\mathbf{z=0.0}$'
ax[0].text(0.95, 0.95, textstr, transform=ax[0].transAxes, fontsize=14,
        verticalalignment='top',ha='right')#, bbox=dict(alpha=0.5))

# Second subplot
ax[1].semilogx(kvec, Pkl_hmc_fid[0]/Pkl_hmc_fid[0], label='linear')
ax[1].semilogx(kvec, pk_nl_fid[0]/Pkl_hmc_fid[0], label='BACCO', c='k')
for i in range(len(kIR2)):
    for j in range(len(cutoff_jn2)):
        for r in range(1):
            ax[1].semilogx(kvec, P0vec2[i][j][r]/Pkl_hmc_fid[0], ls=ls[i], c=c[i], alpha=0.8, label=label_cutoff[i])
            
ax[1].axvline(kv_fid[0]/M_fid.h(), ls=':', c='k', label=r'$\sigma_\mathrm{v}^{-1}$')
ax[1].axvline(knl_fid[0]/M_fid.h(), ls='--', c='k', label=r'$r_\mathrm{nl}^{-1}$')
ax[1].legend(loc='upper left',bbox_to_anchor=(1.01,1.0))
ax[1].set_xlabel(r'$k \,\,\,\,[h/\mathrm{Mpc}]$', fontsize=21)
ax[1].set_xlim([0.01, 4])
ax[1].set_ylim([-0.1, 2.1])
ax[1].set_ylabel(r'$\frac{P^\mathrm{X}(k)}{P^\mathrm{Lin}(k)}$', fontsize=28)

# Save the figure
fig.savefig(os.path.join(FIG_DIR, 'test_cutoff_and_jn.png'), bbox_inches='tight')


In [ ]:
Pk_2h = Pkl_hmc_fid
Pk_2h_LPT = Pk_lpt_fid
Pk_1h = Pk_whmc_halo-Pkl_hmc_fid
Pk_1hdiff = Pk_whmc_halominussphere-Pkl_hmc_fid
Pk_1fdiff = Pk_whmc_filaminussphere-Pkl_hmc_fid
Pk_1sdiff = Pk_whmc_sheetminussphere-Pkl_hmc_fid
Pk_whmc_tweaked = Pk_whmc_full_tweaked-Pkl_hmc_fid
Pk_2h_bacco = np.zeros((3,len(redshifts),len(kvec)))
Pk_2h_LPT_bacco = np.zeros((3,len(redshifts),len(kvec)))
Pk_1h_bacco = np.zeros((3,len(redshifts),len(kvec)))
Pk_1hdiff_bacco = np.zeros((3,len(redshifts),len(kvec)))
Pk_1fdiff_bacco = np.zeros((3,len(redshifts),len(kvec)))
Pk_1sdiff_bacco = np.zeros((3,len(redshifts),len(kvec)))
Pk_whmc_tweaked_bacco = np.zeros((3,len(redshifts),len(kvec)))
for ib in range(3):
    Pk_2h_bacco[ib] = Pkl_hmc_bacco[ib]
    Pk_2h_LPT_bacco[ib] = Pk_lpt_bacco[ib]
    Pk_1h_bacco[ib] = Pk_whmc_halo_bacco[ib]-Pkl_hmc_bacco[ib]
    Pk_1hdiff_bacco[ib] = Pk_whmc_halominussphere_bacco[ib]-Pkl_hmc_bacco[ib]
    Pk_1fdiff_bacco[ib] = Pk_whmc_filaminussphere_bacco[ib]-Pkl_hmc_bacco[ib]
    Pk_1sdiff_bacco[ib] = Pk_whmc_sheetminussphere_bacco[ib]-Pkl_hmc_bacco[ib]
    Pk_whmc_tweaked_bacco[ib] = Pk_whmc_full_tweaked_bacco[ib]-Pkl_hmc_bacco[ib]

# 'thin web' variants (upper edges of the coloured bands in Fig. 4)
Pk_1hdiff_concnfw = Pk_whmc_halominussphere_concnfw-Pkl_hmc_fid
Pk_1fdiff_concnfw = Pk_whmc_filaminussphere_concnfw-Pkl_hmc_fid
Pk_1sdiff_concnfw = Pk_whmc_sheetminussphere_concnfw-Pkl_hmc_fid
Pk_whmc_tweaked_concnfw = Pk_whmc_full_tweaked_concnfw-Pkl_hmc_fid
Pk_1hdiff_bacco_concnfw = np.zeros((3,len(redshifts),len(kvec)))
Pk_1fdiff_bacco_concnfw = np.zeros((3,len(redshifts),len(kvec)))
Pk_1sdiff_bacco_concnfw = np.zeros((3,len(redshifts),len(kvec)))
Pk_whmc_tweaked_bacco_concnfw = np.zeros((3,len(redshifts),len(kvec)))
for ib in range(3):
    Pk_1hdiff_bacco_concnfw[ib] = Pk_whmc_halominussphere_bacco_concnfw[ib]-Pkl_hmc_bacco[ib]
    Pk_1fdiff_bacco_concnfw[ib] = Pk_whmc_filaminussphere_bacco_concnfw[ib]-Pkl_hmc_bacco[ib]
    Pk_1sdiff_bacco_concnfw[ib] = Pk_whmc_sheetminussphere_bacco_concnfw[ib]-Pkl_hmc_bacco[ib]
    Pk_whmc_tweaked_bacco_concnfw[ib] = Pk_whmc_full_tweaked_bacco_concnfw[ib]-Pkl_hmc_bacco[ib]


In [ ]:
Pk_2h_Marcos = Pkl_hmc_Marcos
Pk_2h_LPT_Marcos = Pk_lpt_Marcos
Pk_1h_Marcos = Pk_whmc_halo_Marcos-Pkl_hmc_Marcos
Pk_1hori_Marcos = Pk_ori_Marcos-Pkl_hmc_Marcos
Pk_1horitweaked_Marcos = Pk_ori_tweaked_Marcos-Pkl_hmc_Marcos
Pk_1hs_Marcos = Pk_whmc_halosphere_Marcos-Pkl_hmc_Marcos
Pk_1hdiff_Marcos = Pk_whmc_halominussphere_Marcos-Pkl_hmc_Marcos
Pk_1fdiff_Marcos = Pk_whmc_filaminussphere_Marcos-Pkl_hmc_Marcos
Pk_1sdiff_Marcos = Pk_whmc_sheetminussphere_Marcos-Pkl_hmc_Marcos
Pk_whmc_tweaked_Marcos = Pk_whmc_full_tweaked_Marcos-Pkl_hmc_Marcos # P1s-Ppt + P1f-P1s + P1h-P1f
Pk_whmc_tweaked_Marcos_no1h = Pk_whmc_full_tweaked_Marcos-Pk_whmc_halominussphere_Marcos # P1s-Ppt + P1f-P1s
Pk_whmc_tweaked_Marcos_no1hwithcomp = Pk_whmc_full_tweaked_Marcos-Pk_whmc_halo_Marcos
Pk_1hcompensation_Marcos = Pk_whmc_halominussphere_Marcos-Pk_ori_Marcos
Pk_1hdiff_Marcos_warren = Pk_whmc_halominussphere_Marcos_warren-Pkl_hmc_Marcos
Pk_whmc_tweaked_Marcos_warren = Pk_whmc_full_tweaked_Marcos_warren-Pkl_hmc_Marcos


In [ ]:
# Cache the fiducial / BACCO-simulation decomposition used by Fig. 4 (data/fiducial_bacco/); not needed to run the notebook.
if WRITE["camb"]:
    _d = 'data/fiducial_bacco/'
    for _name, _arr in [('Pk_2h', Pk_2h), ('Pk_1hdiff', Pk_1hdiff), ('Pk_1fdiff', Pk_1fdiff), ('Pk_1sdiff', Pk_1sdiff),
                        ('Pk_whmc_tweaked_kNL1', Pk_whmc_tweaked), ('Pk_2h_LPT_kNL1', Pk_2h_LPT),
                        ('Pk_1hdiff_concnfw', Pk_1hdiff_concnfw), ('Pk_1fdiff_concnfw', Pk_1fdiff_concnfw),
                        ('Pk_1sdiff_concnfw', Pk_1sdiff_concnfw), ('Pk_whmc_tweaked_kNL1_concnfw', Pk_whmc_tweaked_concnfw)]:
        np.savetxt(_d+_name+'.txt', np.c_[kvec, _arr[0], _arr[1], _arr[2]])
    np.savetxt(_d+'pk_nl.txt', np.c_[kvec, pk_nl_fid[0], pk_nl_fid[1], pk_nl_fid[2]])
    for ib in range(3):
        for _name, _arr in [('Pk_2h_bacco', Pk_2h_bacco), ('Pk_1hdiff_bacco', Pk_1hdiff_bacco), ('Pk_1fdiff_bacco', Pk_1fdiff_bacco),
                            ('Pk_1sdiff_bacco', Pk_1sdiff_bacco), ('Pk_whmc_tweaked_bacco_kNL1', Pk_whmc_tweaked_bacco), ('Pk_2h_LPT_bacco_kNL1', Pk_2h_LPT_bacco),
                            ('Pk_1hdiff_bacco_concnfw', Pk_1hdiff_bacco_concnfw), ('Pk_1fdiff_bacco_concnfw', Pk_1fdiff_bacco_concnfw),
                            ('Pk_1sdiff_bacco_concnfw', Pk_1sdiff_bacco_concnfw), ('Pk_whmc_tweaked_bacco_concnfw_kNL1', Pk_whmc_tweaked_bacco_concnfw)]:
            np.savetxt(_d+_name+'_%d.txt'%(ib), np.c_[kvec, _arr[ib][0], _arr[ib][1], _arr[ib][2]])
        np.savetxt(_d+'pk_nl_bacco_%d.txt'%(ib), np.c_[kvec, pk_nl_bacco[ib][0], pk_nl_bacco[ib][1], pk_nl_bacco[ib][2]])


## 7. WHM spectra & paper figures

In [ ]:
# Warren-HMF Marcos spectra (computed in section 5 with WHM_hmf=1); cached under data/Pk_Marcos/ for reference.
if WRITE["camb"]:
    for _name, _arr in [('Pk_1hdiff_Marcos_warren', Pk_1hdiff_Marcos_warren), ('Pk_whmc_tweaked_Marcos_warren', Pk_whmc_tweaked_Marcos_warren),
                        ('Pk_whmc_halo_Marcos_warren', Pk_whmc_halo_Marcos_warren)]:
        np.savetxt(os.path.join(DATA_DIR, 'Pk_Marcos', _name+'.txt'), _arr)


In [ ]:
# esthetic definitions for the plots
font = {'size'   : 18, 'family':'STIXGeneral'}
axislabelfontsize='large'
matplotlib.rc('font', **font)
matplotlib.rc('text', usetex=True)
plt.rcParams['legend.fontsize']='medium'
plt.rcParams["figure.figsize"] = [16.0,10.0]

mode = 'BACCO'
SN = (1440/1080)**3*np.array([0.6,0.44,0.3])

fig, ax = plt.subplots(4,2,sharex=True,gridspec_kw={'height_ratios':[3,1,1,1],'hspace':0.1,'wspace':0.1})
alpha=0.73
i=0

P1h_analytical_Marcos_int = Pk1h_Marcos_int

hmcolor = 'C1'
whmcolor = 'C0'

l1, = ax[0,0].loglog(kvec_Marcos,Pknl_Marcos,c='k',ls='-',label='$P^\mathrm{NL}(k)$')
l2, = ax[0,0].loglog(kvec_Marcos,Pknl_Marcos-P1h_analytical_Marcos_int,c='k',ls='--',label='$P^\mathrm{2H}(k)$')
l3, = ax[0,0].loglog(kvec_Marcos,P1h_analytical_Marcos_int,c='k',ls=':',label='$P^\mathrm{1H}(k)$')
l4, = ax[0,0].loglog(kvec_Marcos,Pk_ori_Marcos[i],c=hmcolor,ls='-',lw=1,label='Original Halo Model')
ax[0,0].loglog(kvec_Marcos,Pk_2h_Marcos[i],c=hmcolor,ls='--')
l5, = ax[0,0].loglog(kvec_Marcos,Pk_2h_Marcos[i]+Pk_1sdiff_Marcos[i]+Pk_1fdiff_Marcos[i]+Pk_1hdiff_Marcos[i],lw=1,c=whmcolor,label=r'WHM (baseline)')
l7, = ax[0,0].loglog(kvec_Marcos,Pk_2h_Marcos[i]+Pk_1sdiff_Marcos[i]+Pk_1fdiff_Marcos[i]+Pk_1hdiff_Marcos_warren[i],lw=1,c='purple',label=r'WHM (Warren 2006)')
ax[0,0].loglog(kvec_Marcos,2*Pk_2h_Marcos[i]+Pk_1sdiff_Marcos[i]+Pk_1fdiff_Marcos[i]+Pk_1hdiff_Marcos[i]-Pk_whmc_halo_Marcos[i],c=whmcolor,ls='--')
ax[0,0].loglog(kvec_Marcos,Pk_whmc_halo_Marcos[i]-Pk_2h_Marcos[i],c=whmcolor,ls=':')
ax[0,0].loglog(kvec_Marcos,Pk_whmc_halo_Marcos_warren[i]-Pk_2h_Marcos[i],c='purple',ls=':')

l8, = ax[0,1].loglog(kvec_Marcos,Pknl_Marcos,c='k',ls='-',label='Measurement')
ax[0,1].loglog(kvec_Marcos,Pknl_Marcos-P1h_analytical_Marcos_int,c='k',ls='--',label='$P^\mathrm{2H}(k)$')
ax[0,1].loglog(kvec_Marcos,P1h_analytical_Marcos_int,c='k',ls=':',label='$P^\mathrm{1H}(k)$')
ax[0,1].loglog(kvec_Marcos,Pk_2h_LPT_Marcos[i]+Pk_ori_Marcos[i]-Pk_2h_Marcos[i],c=hmcolor,ls='-',lw=1,label='HM (theory)')
ax[0,1].loglog(kvec_Marcos,Pk_2h_LPT_Marcos[i],c=hmcolor,ls='--')
ax[0,1].loglog(kvec_Marcos,Pk_2h_LPT_Marcos[i]+Pk_whmc_tweaked_Marcos[i],lw=1,c=whmcolor,label=r'WHM (baseline)')
ax[0,1].loglog(kvec_Marcos,Pk_2h_LPT_Marcos[i]+Pk_whmc_tweaked_Marcos_warren[i],lw=1,c='purple',label=r'WHM (Warren06)')
ax[0,1].loglog(kvec_Marcos,Pk_2h_LPT_Marcos[i]+Pk_whmc_tweaked_Marcos_no1hwithcomp[i],c=whmcolor,ls='--')
ax[0,1].loglog(kvec_Marcos,Pk_whmc_halo_Marcos[i]-Pk_2h_Marcos[i],c=whmcolor,ls=':')
ax[0,1].loglog(kvec_Marcos,Pk_whmc_halo_Marcos_warren[i]-Pk_2h_Marcos[i],c='purple',ls=':')

ax[1,0].semilogx(kvec_Marcos,(Pknl_Marcos-Pknl_Marcos)/Pknl_Marcos,c='k',ls='-',label='$P^\mathrm{NL}(k)$')
ax[1,0].semilogx(kvec_Marcos,(Pknl_Marcos-P1h_analytical_Marcos_int-Pknl_Marcos)/Pknl_Marcos,c='k',ls='--',label='$P^\mathrm{2H}(k)$')
ax[1,0].semilogx(kvec_Marcos,(P1h_analytical_Marcos_int-Pknl_Marcos)/Pknl_Marcos,c='k',ls=':',label='$P^\mathrm{1H}(k)$')
ax[1,0].semilogx(kvec_Marcos,(Pk_ori_Marcos[i]-Pknl_Marcos)/Pknl_Marcos,c=hmcolor,ls='-',lw=1,label='HM (theory)')
ax[1,0].semilogx(kvec_Marcos,(Pk_2h_Marcos[i]+Pk_1sdiff_Marcos[i]+Pk_1fdiff_Marcos[i]+Pk_1hdiff_Marcos[i]-Pknl_Marcos)/Pknl_Marcos,lw=1,c=whmcolor,label=r'WHM (baseline)')
ax[1,0].semilogx(kvec_Marcos,(Pk_2h_Marcos[i]+Pk_1sdiff_Marcos[i]+Pk_1fdiff_Marcos[i]+Pk_1hdiff_Marcos_warren[i]-Pknl_Marcos)/Pknl_Marcos,lw=1,c='purple',label=r'WHM (Warren06)')
l6, = ax[1,0].semilogx(kvec_Marcos,(2*Pk_2h_Marcos[i]+Pk_1sdiff_Marcos[i]+Pk_1fdiff_Marcos[i]+Pk_1hdiff_Marcos[i]-Pk_whmc_halo_Marcos[i]+P1h_analytical_Marcos_int-Pknl_Marcos)/Pknl_Marcos,lw=3,c=whmcolor,label=r'\bf WHM (measured $P^\mathrm{1h}$)')

ax[1,1].semilogx(kvec_Marcos,(Pknl_Marcos-Pknl_Marcos)/Pknl_Marcos,c='k',ls='-',label='$P^\mathrm{NL}(k)$')
ax[1,1].semilogx(kvec_Marcos,(Pknl_Marcos-P1h_analytical_Marcos_int-Pknl_Marcos)/Pknl_Marcos,c='k',ls='--',label='$P^\mathrm{2H}(k)$')
ax[1,1].semilogx(kvec_Marcos,(P1h_analytical_Marcos_int-Pknl_Marcos)/Pknl_Marcos,c='k',ls=':',label='$P^\mathrm{1H}(k)$')
ax[1,1].semilogx(kvec_Marcos,(Pk_2h_LPT_Marcos[i]+Pk_ori_Marcos[i]-Pk_2h_Marcos[i]-Pknl_Marcos)/Pknl_Marcos,c=hmcolor,ls='-',lw=1,label='HM')
ax[1,1].semilogx(kvec_Marcos,(Pk_2h_LPT_Marcos[i]+Pk_whmc_tweaked_Marcos[i]-Pknl_Marcos)/Pknl_Marcos,lw=1,c=whmcolor,label=r'WHM (baseline)')
ax[1,1].semilogx(kvec_Marcos,(Pk_2h_LPT_Marcos[i]+Pk_whmc_tweaked_Marcos_warren[i]-Pknl_Marcos)/Pknl_Marcos,lw=1,c='purple',label=r'WHM (Warren06)')
ax[1,1].semilogx(kvec_Marcos,(Pk_2h_LPT_Marcos[i]+Pk_whmc_tweaked_Marcos_no1hwithcomp[i]+P1h_analytical_Marcos_int-Pknl_Marcos)/Pknl_Marcos,lw=3,c=whmcolor,label=r'\bf WHM (measured $P^\mathrm{1h}$)')

ax[2,0].semilogx(kvec_Marcos,(Pknl_Marcos-(Pknl_Marcos-P1h_analytical_Marcos_int))/(Pknl_Marcos-P1h_analytical_Marcos_int),c='k',ls='-',label='$P^\mathrm{NL}(k)$')
ax[2,0].semilogx(kvec_Marcos,(Pknl_Marcos-P1h_analytical_Marcos_int-(Pknl_Marcos-P1h_analytical_Marcos_int))/(Pknl_Marcos-P1h_analytical_Marcos_int),c='k',ls='--',label='$P^\mathrm{2H}(k)$')
ax[2,0].semilogx(kvec_Marcos,(P1h_analytical_Marcos_int-(Pknl_Marcos-P1h_analytical_Marcos_int))/(Pknl_Marcos-P1h_analytical_Marcos_int),c='k',ls=':',label='$P^\mathrm{1H}(k)$')
ax[2,0].semilogx(kvec_Marcos,(Pk_2h_Marcos[i]-(Pknl_Marcos-P1h_analytical_Marcos_int))/(Pknl_Marcos-P1h_analytical_Marcos_int),c=hmcolor,ls='--')
ax[2,0].semilogx(kvec_Marcos,(2*Pk_2h_Marcos[i]+Pk_1sdiff_Marcos[i]+Pk_1fdiff_Marcos[i]+Pk_1hdiff_Marcos[i]-Pk_whmc_halo_Marcos[i]-(Pknl_Marcos-P1h_analytical_Marcos_int))/(Pknl_Marcos-P1h_analytical_Marcos_int),c=whmcolor,ls='--')

ax[2,1].semilogx(kvec_Marcos,(Pknl_Marcos-(Pknl_Marcos-P1h_analytical_Marcos_int))/(Pknl_Marcos-P1h_analytical_Marcos_int),c='k',ls='-',label='$P^\mathrm{NL}(k)$')
ax[2,1].semilogx(kvec_Marcos,(Pknl_Marcos-P1h_analytical_Marcos_int-(Pknl_Marcos-P1h_analytical_Marcos_int))/(Pknl_Marcos-P1h_analytical_Marcos_int),c='k',ls='--',label='$P^\mathrm{2H}(k)$')
ax[2,1].semilogx(kvec_Marcos,(P1h_analytical_Marcos_int-(Pknl_Marcos-P1h_analytical_Marcos_int))/(Pknl_Marcos-P1h_analytical_Marcos_int),c='k',ls=':',label='$P^\mathrm{1H}(k)$')
ax[2,1].semilogx(kvec_Marcos,(Pk_2h_LPT_Marcos[i]-(Pknl_Marcos-P1h_analytical_Marcos_int))/(Pknl_Marcos-P1h_analytical_Marcos_int),c=hmcolor,ls='--')
ax[2,1].semilogx(kvec_Marcos,(Pk_2h_LPT_Marcos[i]+Pk_whmc_tweaked_Marcos_no1hwithcomp[i]-(Pknl_Marcos-P1h_analytical_Marcos_int))/(Pknl_Marcos-P1h_analytical_Marcos_int),c=whmcolor,ls='--')

ax[3,0].semilogx(kvec_Marcos,(Pknl_Marcos-(P1h_analytical_Marcos_int))/(P1h_analytical_Marcos_int),c='k',ls='-',label='$P^\mathrm{NL}(k)$')
ax[3,0].semilogx(kvec_Marcos,(Pknl_Marcos-P1h_analytical_Marcos_int-(P1h_analytical_Marcos_int))/(P1h_analytical_Marcos_int),c='k',ls='--',label='$P^\mathrm{2H}(k)$')
ax[3,0].semilogx(kvec_Marcos,(P1h_analytical_Marcos_int-(P1h_analytical_Marcos_int))/(P1h_analytical_Marcos_int),c='k',ls=':',label='$P^\mathrm{1H}(k)$')
ax[3,0].semilogx(kvec_Marcos,(Pk_whmc_halo_Marcos[i]-Pk_2h_Marcos[i]-(P1h_analytical_Marcos_int))/(P1h_analytical_Marcos_int),c=whmcolor,ls=':')
ax[3,0].semilogx(kvec_Marcos,(Pk_whmc_halo_Marcos_warren[i]-Pk_2h_Marcos[i]-(P1h_analytical_Marcos_int))/(P1h_analytical_Marcos_int),c='purple',ls=':')

ax[3,1].semilogx(kvec_Marcos,(Pknl_Marcos-(P1h_analytical_Marcos_int))/(P1h_analytical_Marcos_int),c='k',ls='-',label='$P^\mathrm{NL}(k)$')
ax[3,1].semilogx(kvec_Marcos,(Pknl_Marcos-P1h_analytical_Marcos_int-(P1h_analytical_Marcos_int))/(P1h_analytical_Marcos_int),c='k',ls='--',label='$P^\mathrm{2H}(k)$')
ax[3,1].semilogx(kvec_Marcos,(P1h_analytical_Marcos_int-(P1h_analytical_Marcos_int))/(P1h_analytical_Marcos_int),c='k',ls=':',label='$P^\mathrm{1H}(k)$')
ax[3,1].semilogx(kvec_Marcos,(Pk_whmc_halo_Marcos[i]-Pk_2h_Marcos[i]-(P1h_analytical_Marcos_int))/(P1h_analytical_Marcos_int),c=whmcolor,ls=':')
ax[3,1].semilogx(kvec_Marcos,(Pk_whmc_halo_Marcos_warren[i]-Pk_2h_Marcos[i]-(P1h_analytical_Marcos_int))/(P1h_analytical_Marcos_int),c='purple',ls=':')

ax[0,1].set_ylim(ax[0,0].get_ylim())
for case in range(0,2):
    ax[0,case].axvline(kv_Marcos[i]/0.6766,c='magenta',ls=':')
    ax[0,case].axvline(knl_Marcos[i]/0.6766,c='magenta',ls='--')
    for j in range(1,4):
        ax[j,case].fill_between(kvec_Marcos, (Pk_hmc_Marcos[i]-Pk_hmc_Marcos[i])/Pk_hmc_Marcos[i]-0.05, (Pk_hmc_Marcos[i]-Pk_hmc_Marcos[i])/Pk_hmc_Marcos[i]+0.05, alpha=0.1, color='k')
        ax[j,case].set_ylim([-0.225,0.15])
        ax[j,case].set_xlim([0.02,2.0])
        ax[j,case].axvline(kv_Marcos[i]/0.6766,c='magenta',ls=':')
        ax[j,case].axvline(knl_Marcos[i]/0.6766,c='magenta',ls='--')

ax[0,0].set_title(r'Linear PT',fontsize=21)
ax[0,1].set_title(r'Lagrangian PT ($1\ell$-LPT)',fontsize=21)

textstr = r'$\sigma_\mathrm{v}^{-1}(z)$'
ax[0,0].text((np.log10(kv_Marcos[i]/0.6766)-np.log10(0.02))/(np.log10(2)-np.log10(0.02))-0.05, 0.03, textstr, transform=ax[0,0].transAxes, fontsize=18, c='magenta',
        verticalalignment='bottom',ha='right')#, bbox=dict(alpha=0.5))

textstr = r'$r_\mathrm{nl}^{-1}(z)$'
ax[0,0].text((np.log10(knl_Marcos[i]/0.6766)-np.log10(0.02))/(np.log10(2)-np.log10(0.02))+0.05, 0.03, textstr, transform=ax[0,0].transAxes, fontsize=18, c='magenta',
        verticalalignment='bottom',ha='left')#, bbox=dict(alpha=0.5))

ax[0,0].set_ylabel(r'$P^\mathrm{X}(k)$',fontsize=20)
ax[1,0].set_ylabel(r'$\frac{P^\mathrm{X}(k) - P_\mathrm{meas}^\mathrm{NL}(k)}{P_\mathrm{meas}^\mathrm{NL}(k)}$',fontsize=20)
ax[2,0].set_ylabel(r'$\frac{P^\mathrm{X}(k) - P_\mathrm{meas}^\mathrm{2H}(k)}{P_\mathrm{meas}^\mathrm{2H}(k)}$',fontsize=20)
ax[3,0].set_ylabel(r'$\frac{P^\mathrm{X}(k) - P_\mathrm{meas}^\mathrm{1H}(k)}{P_\mathrm{meas}^\mathrm{1H}(k)}$',fontsize=20)

ax[3,0].set_xlabel(r'$k \,\,\,\,[h/\mathrm{Mpc}]$', fontsize=20)
ax[3,1].set_xlabel(r'$k \,\,\,\,[h/\mathrm{Mpc}]$', fontsize=20)

fig.savefig(os.path.join(FIG_DIR, 'theo_vs_measured_p1h_linlag_all.png'),bbox_inches='tight')

handle_list = [l1,l8,l2,l4,l3,l5,l6,l7]
lable_list = []
for i in range(len(handle_list)):
    lable_list.append(handle_list[i].get_label())

fig_leg = plt.figure(figsize=(14.0,1.0))

leg = fig_leg.legend(handle_list, lable_list,
                loc='center',
          ncol=4, borderaxespad=0.,fontsize=16)

plt.axis('off')
plt.show()


In [ ]:
import matplotlib

# esthetic definitions for the plots
font = {'size'   : 18, 'family':'STIXGeneral'}
axislabelfontsize='large'
matplotlib.rc('font', **font)
matplotlib.rc('text', usetex=True)
plt.rcParams['legend.fontsize']='medium'
plt.rcParams["figure.figsize"] = [16.0,10.0]

mode = 'BACCO'
SN = (1440/1080)**3*np.array([0.6,0.44,0.3])

hmcolor = 'C1'
whmcolor = 'C0'

kvec = np.logspace(-3.9999999999,np.log10(4.8),100) # k in h/Mpc
fig, ax = plt.subplots(4,2,sharex=True,gridspec_kw={'height_ratios':[3,1,1,1],'hspace':0.1,'wspace':0.1})
i=0
l1, = ax[0,0].loglog(kvec,pk_nl_fid[0],c='k',ls='-',label='$P^\mathrm{NL}(k)$')
l2, = ax[0,0].loglog(kvec,pk_nl_fid[0],c='k',ls='--',label='$P^\mathrm{2H}(k)$')
l3, = ax[0,0].loglog(kvec,pk_nl_fid[0],c='k',ls=':',label='$P^\mathrm{1H}(k)$')
l4, = ax[0,0].loglog(kvec,pk_nl_fid[0],c=hmcolor,ls='-',lw=1,label='Original Halo Model')
l5, = ax[0,0].loglog(kvec,pk_nl_fid[0],lw=1,c=whmcolor,label=r'WHM (baseline)')
l6, = ax[1,0].semilogx(kvec,pk_nl_fid[0],lw=3,c=whmcolor,label=r'\bf WHM (measured $P^\mathrm{1h}$)')
l7, = ax[0,0].loglog(kvec,pk_nl_fid[0],lw=1,c='purple',label=r'WHM (Warren 2006)')
l8, = ax[0,1].loglog(kvec,pk_nl_fid[0],c='k',ls='-',label='Simulation')

handle_list = [l1,l8,l2,l4,l3,l5,l6,l7]
lable_list = []
for i in range(len(handle_list)):
    lable_list.append(handle_list[i].get_label())

fig_leg = plt.figure(figsize=(14.0,1.0))

leg = fig_leg.legend(handle_list, lable_list,
                loc='center',
          ncol=4, borderaxespad=0.,fontsize=16)

plt.axis('off')
fig_leg.savefig(os.path.join(FIG_DIR, 'theo_vs_measured_p1h_linlag_all_leg_v2.png'),bbox_inches='tight')


In [ ]:
import FlamingoBaryonResponseEmulator as fre
from matplotlib.pyplot import cm
# Load FLAMINGO emulator
emulator = fre.FlamingoBaryonResponseEmulator()

# Specify baryon model
fgas_sigma_low = -10.
fgas_sigma_high = 4.
Mstar_sigma_low = -3.
Mstar_sigma_high = 2.
jet_fraction = 0.

k_min_plot = 0.02
k_max_plot = 4.8

# Prepare figure 
ax = plt.subplot()
ax.set_xscale("log")

colors_z = cm.plasma(np.linspace(0.0, 0.9, len(redshifts)))

response_low = np.zeros((len(redshifts),len(kvec)))
response_high = np.zeros((len(redshifts),len(kvec)))

# Plot emulator data
for i in range(np.size(redshifts)):

    # Call emulator
    response_low[i] = emulator.predict(kvec, redshifts[i], fgas_sigma_low, Mstar_sigma_low, jet_fraction)
    response_high[i] = emulator.predict(kvec, redshifts[i], fgas_sigma_high, Mstar_sigma_high, jet_fraction)

    # Plot response
    ax.plot(kvec, response_low[i], color=colors_z[i],
            label="$z=%.1f$" % redshifts[i],
            lw=2,
            ms=1.5)
    ax.plot(kvec, response_high[i], color=colors_z[i], ls=':',
            label="$z=%.1f$" % redshifts[i],
            lw=2,
            ms=1.5)

ax.set_xlim(k_min_plot, k_max_plot)
ax.set_ylim(0.72, 1.14)


In [ ]:
from matplotlib.ticker import LogLocator, MultipleLocator, ScalarFormatter
from matplotlib.legend_handler import HandlerBase
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.font_manager as fm

# esthetic definitions for the plots
font = {'size'   : 18, 'family':'STIXGeneral'}
axislabelfontsize='large'
matplotlib.rc('font', **font)
matplotlib.rc('text', usetex=True)
plt.rcParams['legend.fontsize']='medium'
plt.rcParams["figure.figsize"] = [18.0,6.0]
# LaTeX preamble to include `bm` package
plt.rcParams['text.latex.preamble'] = r'\usepackage{bm}'

mode = 'BACCO'

ib = 2

hbacco = [0.65, 0.60, 0.70]

fig, ax = plt.subplots(2, 3, sharex=True, sharey=True, gridspec_kw={'height_ratios':[1, 1]})

alpha=0.73
alphalin = 0.5
for i in range(len(redshifts)):
    ax[0,i].semilogx(kvec,(Pk_ori_bacco[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i],label='Halo Model',c='k',ls='--',alpha=alphalin)
    ax[0,i].semilogx(kvec,(Pk_hmc_bacco[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i],label='HMcode-2020',c='k',ls='-',alpha=alphalin)
    ax[0,i].semilogx(kvec,(Pk_2h_bacco[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i],c='grey',label='PT',alpha=alphalin)
    ax[0,i].semilogx(kvec,(Pk_2h_bacco[ib][i]+Pk_1hdiff_bacco[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i],c='C3',alpha=alphalin)
    ax[0,i].semilogx(kvec,(Pk_2h_bacco[ib][i]+Pk_1fdiff_bacco[ib][i]+Pk_1hdiff_bacco[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i],c='C2',alpha=alphalin)
    ax[0,i].semilogx(kvec,(Pk_2h_bacco[ib][i]+Pk_1sdiff_bacco[ib][i]+Pk_1fdiff_bacco[ib][i]+Pk_1hdiff_bacco[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i],lw=3,c='C0',alpha=alphalin)
    ax[0,i].semilogx(kvec,(Pk_2h_bacco[ib][i]+Pk_1hdiff_bacco_concnfw[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i],c='C3',alpha=alphalin)
    ax[0,i].semilogx(kvec,(Pk_2h_bacco[ib][i]+Pk_1fdiff_bacco_concnfw[ib][i]+Pk_1hdiff_bacco_concnfw[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i],c='C2',alpha=alphalin)
    ax[0,i].semilogx(kvec,(Pk_2h_bacco[ib][i]+Pk_1sdiff_bacco_concnfw[ib][i]+Pk_1fdiff_bacco_concnfw[ib][i]+Pk_1hdiff_bacco_concnfw[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i],lw=3,c='C0',alpha=alphalin)
    ax[0,i].fill_between(kvec, 
                         (Pk_2h_bacco[ib][i]+Pk_1hdiff_bacco[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i], 
                         (Pk_2h_bacco[ib][i]+Pk_1hdiff_bacco_concnfw[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i], 
                         alpha=0.1*alphalin, 
                         color='C3')
    ax[0,i].fill_between(kvec, 
                         (Pk_2h_bacco[ib][i]+Pk_1fdiff_bacco[ib][i]+Pk_1hdiff_bacco[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i], 
                         (Pk_2h_bacco[ib][i]+Pk_1fdiff_bacco_concnfw[ib][i]+Pk_1hdiff_bacco_concnfw[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i], 
                         alpha=0.1*alphalin, 
                         color='C2')
    ax[0,i].fill_between(kvec, 
                         (Pk_2h_bacco[ib][i]+Pk_1sdiff_bacco[ib][i]+Pk_1fdiff_bacco[ib][i]+Pk_1hdiff_bacco[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i], 
                         (Pk_2h_bacco[ib][i]+Pk_1sdiff_bacco_concnfw[ib][i]+Pk_1fdiff_bacco_concnfw[ib][i]+Pk_1hdiff_bacco_concnfw[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i], 
                         alpha=0.2*alphalin, 
                         color='C0')

for i in range(len(redshifts)):
    l1, = ax[1,i].semilogx(kvec,(Pk_ori_bacco[ib][i]-Pk_2h_bacco[ib][i]+Pk_2h_LPT_bacco[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i],c='k',ls='--')
    l2, = ax[1,i].semilogx(kvec,(Pk_hmc_bacco[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i],c='k',ls='-')
    l3, = ax[1,i].semilogx(kvec,(Pk_2h_LPT_bacco[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i],c='grey')
    l4, = ax[1,i].semilogx(kvec,(Pk_2h_LPT_bacco[ib][i]+Pk_1hdiff_bacco[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i],label='PT+halo',c='C3')
    ax[1,i].semilogx(kvec,(Pk_2h_LPT_bacco[ib][i]+Pk_1hdiff_bacco_concnfw[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i],label='PT+halo',c='C3')
    l5, = ax[1,i].semilogx(kvec,(Pk_2h_LPT_bacco[ib][i]+Pk_1fdiff_bacco[ib][i]+Pk_1hdiff_bacco[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i],label='PT+fil.+halo',c='C2')
    ax[1,i].semilogx(kvec,(Pk_2h_LPT_bacco[ib][i]+Pk_1fdiff_bacco_concnfw[ib][i]+Pk_1hdiff_bacco_concnfw[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i],label='PT+fil.+halo',c='C2')
    l6, = ax[1,i].semilogx(kvec,(Pk_2h_LPT_bacco[ib][i]+Pk_whmc_tweaked_bacco[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i],lw=3,label='PT+sheet+fil.+halo',c='C0')
    ax[1,i].semilogx(kvec,(Pk_2h_LPT_bacco[ib][i]+Pk_whmc_tweaked_bacco_concnfw[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i],lw=3,label='PT+sheet+fil.+halo',c='C0')
    ax[1,i].fill_between(kvec, 
                         (Pk_2h_LPT_bacco[ib][i]+Pk_1hdiff_bacco[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i], 
                         (Pk_2h_LPT_bacco[ib][i]+Pk_1hdiff_bacco_concnfw[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i], 
                         alpha=0.1, 
                         color='C3')
    ax[1,i].fill_between(kvec, 
                         (Pk_2h_LPT_bacco[ib][i]+Pk_1fdiff_bacco[ib][i]+Pk_1hdiff_bacco[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i], 
                         (Pk_2h_LPT_bacco[ib][i]+Pk_1fdiff_bacco_concnfw[ib][i]+Pk_1hdiff_bacco_concnfw[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i], 
                         alpha=0.1, 
                         color='C2')
    ax[1,i].fill_between(kvec, 
                         (Pk_2h_LPT_bacco[ib][i]+Pk_whmc_tweaked_bacco[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i], 
                         (Pk_2h_LPT_bacco[ib][i]+Pk_whmc_tweaked_bacco_concnfw[ib][i]-pk_nl_bacco[ib][i])/pk_nl_bacco[ib][i], 
                         alpha=0.2, 
                         color='C0')
for i in range(len(redshifts)):
    ax[0,i].fill_between(kvec, (Pk_hmc_bacco[ib][i]-Pk_hmc_bacco[ib][i])/Pk_hmc_bacco[ib][i]-0.01, (Pk_hmc_bacco[ib][i]-Pk_hmc_bacco[ib][i])/Pk_hmc_bacco[ib][i]+0.01, alpha=0.2*alphalin, color='k')
    ax[0,i].fill_between(kvec, (Pk_hmc_bacco[ib][i]-Pk_hmc_bacco[ib][i])/Pk_hmc_bacco[ib][i]-0.05, (Pk_hmc_bacco[ib][i]-Pk_hmc_bacco[ib][i])/Pk_hmc_bacco[ib][i]+0.05, alpha=0.1*alphalin, color='k')
    ax[0,i].set_ylim([-0.425,0.15])
    ax[0,i].set_xlim([0.01,4.8])
    ax[0,i].set_title(r'$z=%.1f$'%(redshifts[i]),fontsize=21)
    ax[1,i].fill_between(kvec, (Pk_hmc_bacco[ib][i]-Pk_hmc_bacco[ib][i])/Pk_hmc_bacco[ib][i]-0.01, (Pk_hmc_bacco[ib][i]-Pk_hmc_bacco[ib][i])/Pk_hmc_bacco[ib][i]+0.01, alpha=0.2, color='k')
    ax[1,i].fill_between(kvec, (Pk_hmc_bacco[ib][i]-Pk_hmc_bacco[ib][i])/Pk_hmc_bacco[ib][i]-0.05, (Pk_hmc_bacco[ib][i]-Pk_hmc_bacco[ib][i])/Pk_hmc_bacco[ib][i]+0.05, alpha=0.1, color='k')
    ax[1,i].set_ylim([-0.425,0.15])
    ax[1,i].set_xlim([0.01,4.8])
    ax[0,i].axvline(kv_bacco[ib][i]/hbacco[ib],c='magenta',ls=':')
    ax[0,i].axvline(knl_bacco[ib][i]/hbacco[ib],c='magenta',ls='--')
    ax[1,i].axvline(kv_bacco[ib][i]/hbacco[ib],c='magenta',ls=':')
    ax[1,i].axvline(knl_bacco[ib][i]/hbacco[ib],c='magenta',ls='--')
    ax[0,i].fill_between(kvec, response_low[i]-1, response_high[i]-1, alpha=0.2*alphalin, color='darkorange')
    ax[1,i].fill_between(kvec, response_low[i]-1, response_high[i]-1, alpha=0.2, color='darkorange')

textstr = r'Linear PT'
ax[0,0].text(0.03, 0.03, textstr, transform=ax[0,0].transAxes, fontsize=21,
        verticalalignment='bottom',ha='left')#, bbox=dict(alpha=0.5))
textstr = r'\bm{$\bm{1\ell}$\textbf{-LPT}'
ax[1,0].text(0.03, 0.03, textstr, transform=ax[1,0].transAxes, fontsize=21,
        verticalalignment='bottom',ha='left')#, bbox=dict(alpha=0.5))

textstr = r'$\sigma_\mathrm{v}^{-1}(z)$'
ax[0,1].text((np.log10(kv_bacco[ib][1]/hbacco[ib])-np.log10(0.01))/(np.log10(4.8)-np.log10(0.01))-0.03, 0.03, textstr, transform=ax[0,1].transAxes, fontsize=21, c='magenta',
        verticalalignment='bottom',ha='right')#, bbox=dict(alpha=0.5))

textstr = r'$r_\mathrm{nl}^{-1}(z)$'
ax[0,1].text((np.log10(knl_bacco[ib][1]/hbacco[ib])-np.log10(0.01))/(np.log10(4.8)-np.log10(0.01))+0.03, 0.03, textstr, transform=ax[0,1].transAxes, fontsize=21, c='magenta',
        verticalalignment='bottom',ha='left')#, bbox=dict(alpha=0.5))

blue_patch = mpatches.Patch(label=r'$\bm{Baryon~feedback}$', linewidth=1, ec=(0,0,0,0.8), fc=(0,0,1,0.2), ls='-')
yellow_patch = mpatches.Patch(label=r'$\bm{\mathbf{Baryon~feedback}}$', linewidth=1, color='darkorange', alpha=0.5, ls='-')

prop_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

# Custom legend handler to display gradient with transparency
class HandlerColorCycleGradient(HandlerBase):
    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
        # Define the specific colors and their transparencies
        colors = ['C3', 'C2', 'C0']
        alphas = [0.5, 0.5, 0.5]
        
        num_gradient_steps = 256
        num_colors = len(colors)
        step_size = num_gradient_steps // num_colors
        gradient = np.zeros((1, num_gradient_steps, 4))  # RGBA gradient

        for i, (color, alpha) in enumerate(zip(colors, alphas)):
            rgba_color = (*mcolors.to_rgb(color), alpha)
            gradient[:, i * step_size:(i + 1) * step_size, :] = rgba_color

        gradient_image = np.tile(gradient, (10, 1, 1))  # Stack vertically for visibility

        # Display the gradient in the legend only
        img = legend.axes.imshow(gradient_image, aspect='auto',
                                 extent=(xdescent, width, ydescent, height), transform=trans)
        
        # Immediately remove the gradient from the axis (it remains in the legend)
        img.remove()
        
        return [img]

# Create a dummy handle and add it to the legend with the custom handler
gradient_patch = mpatches.Patch(label=r'$\bm{W_\mathrm{s}~\mathrm{and}~W_\mathrm{f}}$')
# '$w_0 w_a$CDM+$\sum m_\nu$ cosmologies'
ax[1,2].legend(handles=[yellow_patch,gradient_patch], handler_map={gradient_patch: HandlerColorCycleGradient()},frameon=False, 
               loc='lower left', bbox_to_anchor=(0.0, 0.0))

textstr = r'$\mathbf{Uncertainties~due~to:}$'
ax[1,2].text(0.035, 0.45,  textstr, transform=ax[1,2].transAxes, c='k',
        verticalalignment='bottom',ha='left')#, bbox=dict(alpha=0.5))

grey_patch_inner = mpatches.Patch(label=r'$\bm{1\%}$', linewidth=1, color='k', alpha=0.3, ls='-')
grey_patch_outer = mpatches.Patch(label=r'$\bm{5\%}$', linewidth=1, color='k', alpha=0.1, ls='-')
ax[1,1].legend(handles=[grey_patch_outer,grey_patch_inner],frameon=False, 
               loc='lower left', bbox_to_anchor=(0.0, 0.0))

textstr = r'$\mathbf{Accuracy~floor:}$'
ax[1,1].text(0.035, 0.45,  textstr, transform=ax[1,1].transAxes, c='k',
        verticalalignment='bottom',ha='left')#, bbox=dict(alpha=0.5))

ax[0,0].set_ylabel(r'$\frac{P^\mathrm{X}(k) - P^\mathrm{NL}(k)}{P^\mathrm{NL}(k)}$',fontsize=28)
ax[1,0].set_ylabel(r'$\bm{\frac{P^\mathrm{X}(k) - P^\mathrm{NL}(k)}{P^\mathrm{NL}(k)}}$',fontsize=28)

ax[1,0].set_xlabel(r'$\bm{k \,\,\,\, [h/\mathrm{Mpc}]}$', fontsize=21)
ax[1,1].set_xlabel(r'$\bm{k \,\,\,\, [h/\mathrm{Mpc}]}$', fontsize=21)
ax[1,2].set_xlabel(r'$\bm{k \,\,\,\, [h/\mathrm{Mpc}]}$', fontsize=21)

# Make the frames of the lower row thicker
for axes in ax[1, :]:
    for spine in axes.spines.values():
        spine.set_linewidth(2)  # Adjust the width as needed
    # Enable and thicken major and minor ticks
    axes.tick_params(axis='both', which='both', width=2)  # Major and minor
    axes.minorticks_on()  # Enable minor ticks

    # Optionally, you can adjust the length of the ticks if needed
    axes.tick_params(axis='both', which='major', length=6)
    axes.tick_params(axis='both', which='minor', length=3)

# Get the bounding box of the subplot in figure coordinates
for axes in ax[0, :]:
    bbox = axes.get_position()
    
    # Convert the bbox to inches using the figure's DPI
    width = bbox.width * fig.get_figwidth()
    height = bbox.height * fig.get_figheight()

    print(f"Width: {width} inches, Height: {height} inches")

# Get the bounding box of the subplot in figure coordinates
for axes in ax[1, :]:
    bbox = axes.get_position()
    
    # Convert the bbox to inches using the figure's DPI
    width = bbox.width * fig.get_figwidth()
    height = bbox.height * fig.get_figheight()

    print(f"Width: {width} inches, Height: {height} inches")

fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'WHMcode_spectra_comp_kNL1_concnfw_narya.png'))

fig_leg = plt.figure(figsize=(14,1.0))

leg = fig_leg.legend((l2,l1,l3,l4,l5,l6),(
                                          r'$\mathtt{HMcode-2020}$: \quad ~~$(P^\mathrm{X} = P^\mathtt{HMcode-2020})$',
                                          r'Orig. Halo Model: $(P^\mathrm{X} = P^\mathrm{PT}+P^\mathrm{1h})$',
                                          r'PT: $\quad~(P^\mathrm{X} = P^\mathrm{PT})$',
                                          r'Halo: $(P^\mathrm{X} = P^\mathrm{PT}+\widehat{P}^\mathrm{1h})$',
                                          r'Halo+Fil: $~\,(P^\mathrm{X} = P^\mathrm{PT}+\widehat{P}^\mathrm{1f}+\widehat{P}^\mathrm{1h})$',
                                          r'\bf WHM: $\bm{(P^\mathrm{X} = P^\mathrm{PT}+\widehat{P}^\mathrm{1s}+\widehat{P}^\mathrm{1f}+\widehat{P}^\mathrm{1h})}$'),
                loc='center',
          ncol=3, borderaxespad=0.,fontsize=16)

plt.axis('off')
fig_leg.savefig(os.path.join(FIG_DIR, 'WHMcode_spectra_comp_leg_wide.png'))

plt.show()


In [ ]:
import numpy as np
from matplotlib.ticker import ScalarFormatter, LogLocator, NullFormatter, FormatStrFormatter
from math import factorial
# esthetic definitions for the plots
font = {'size'   : 24, 'family':'STIXGeneral'}
axislabelfontsize='HUGE'
matplotlib.rc('font', **font)
matplotlib.rc('text', usetex=True)
plt.rcParams['legend.fontsize']='medium'
plt.rcParams["figure.figsize"] = [8.0,6.0]

# Define the functions
def massfunc(x, a, alpha, beta, norm):
    term1 = np.exp(-a*x**2 * (1 + beta*(a*x**2)**(-alpha))**2 / 2)
    term2 = (1 + beta/(a*x**2)**alpha * 
             (1 - alpha + (alpha*(alpha - 1)/factorial(2)) - 
              (alpha*(alpha - 1)*(alpha - 2)/factorial(3)) + 
              (alpha*(alpha - 1)*(alpha - 2)*(alpha - 3)/factorial(4)) - 
              (alpha*(alpha - 1)*(alpha - 2)*(alpha - 3)*(alpha - 4)/factorial(5))))
    return norm * term1 * term2

def halomassfunc99(x, a, p):
    return 0.21616 * (1 + (a*x**2)**(-p)) * np.exp(-a*x**2 / 2)

# Define the x values
x = np.logspace(np.log10(0.05), np.log10(4.0), 1000)

# Calculate y values for each function
y1 = massfunc(x, 0.707, 0.55, -0.56, 0.631) * x
y2 = massfunc(x, 0.707, 0.28, -0.012, 0.672) * x
y3 = massfunc(x, 0.707, 0.61, 0.45, 0.639) * x
y4 = halomassfunc99(x, 0.707, 0.3) * x
y5 = halomassfunc99(x, 1, 0) / 0.21616 * 0.5 * x
y6 = HMF_Warren(x, 0.7234, 1.625, 0.2538, 1.1982, M_Marcos.Omega_m())

# Create the plot
plt.figure(figsize=(8, 6))
plt.loglog(x, y1, label=r'Sheets', color='C0', lw=2)
plt.loglog(x, y2, label=r'Filaments', color='C2', lw=2)
plt.loglog(x, y3, label=r'Halos', color='C1', lw=2)
plt.loglog(x, y4, label=r'Sheth \& Tormen', color='C3', lw=2)
plt.loglog(x, y5, label=r'Press \& Schechter', color='k', lw=2)
plt.loglog(x, y6, label=r'Warren et al.', color='purple', lw=2, ls='--')

plt.xlim(0.05,4)
plt.ylim(0.08,1.05)

# Customize ticks
ax = plt.gca()
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.xaxis.get_major_formatter().set_scientific(False)
ax.yaxis.get_major_formatter().set_scientific(False)
ax.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))
ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

# Set minor ticks
ax.xaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(1, 10) * 0.1, numticks=10))
ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(1, 10) * 0.1, numticks=10))

# Set specific minor tick labels
minor_ticks_x = [0.05, 0.5]
minor_ticks_y = [0.05, 0.5]
ax.xaxis.set_minor_formatter(NullFormatter())
ax.yaxis.set_minor_formatter(NullFormatter())
ax.xaxis.set_minor_formatter(plt.FuncFormatter(lambda val, pos: f"{val}" if val in minor_ticks_x else ""))
ax.yaxis.set_minor_formatter(plt.FuncFormatter(lambda val, pos: f"{val}" if val in minor_ticks_y else ""))

# Add labels and legend
plt.xlabel(r'$\nu \equiv \delta_\mathrm{crit}/\sigma(M)$', fontsize=24)
plt.ylabel(r'$\nu \cdot f(\nu)$', fontsize=24)
plt.legend(loc='best',fontsize=18)

plt.savefig(os.path.join(FIG_DIR, 'halomassfunctions.png'), bbox_inches='tight')
plt.show()


In [ ]:
# deal with the CAMB runs
cp = {}
CambRun = {}
# (halofit_version, extra set_params) per case; W = baseline WHM settings. The Warren-HMF (WHM_hmf=1) and HMcode-original runs are only needed for Fig. 5 (BACCO sample).
if SAMPLE == 'BACCO':
    halofit = [(9,{}), (6,{}), (14,W), (14,dict(W, WHM_hmf=1))]
else:
    halofit = [(9,{}), (14,W)]
for i in (range(0,Ncosmo) if RECOMPUTE["camb"] else range(0)):
    cp[i] = {}
    CambRun[i] = {}
    for j in range(0,len(halofit)):
        cp[i][j] = camb.CAMBparams()
        cp[i][j].set_cosmology(H0=hubble[i]*100,
                         omch2=om_cdm[i],
                         ombh2=om_b[i],
                         YHe=YHe,
                         TCMB=T_CMB,
                        )
        cp[i][j].InitPower.set_params(ns=n_s[i], As=A_s[i])#*s8_corr[i])
        cp[i][j].scalar_initial_condition=1
        if (want_varying_neutrino_mass):
            cp[i][j].omnuh2=Mnu[i]/93.14
            cp[i][j].num_nu_massive=1
            cp[i][j].nu_mass_degeneracies=[CFG['camb_nu_deg']]
            cp[i][j].num_nu_massless=CFG['camb_nu_massless']
            
        if (want_varying_dark_energy):
            cp[i][j].DarkEnergy=camb.DarkEnergyPPF()
            cp[i][j].DarkEnergy.set_params(w=w0[i], wa=wa[i])

        cp[i][j].WantTransfer = True
        cp[i][j].WantCls = False
        cp[i][j].Want_CMB = False
        cp[i][j].Want_CMB_lensing = False
        cp[i][j].DoLensing = False
        cp[i][j].AccurateReionization = False

        cp[i][j].NonLinear = camb.model.NonLinear_both
        cp[i][j].NonLinearModel.set_params(halofit_version=halofit[j][0], **halofit[j][1])
        cp[i][j].set_matter_power(redshifts=np.linspace(0,10,num=100),
                            kmax=kmax_for_computation,
                            accurate_massive_neutrino_transfers=True)

        if (want_output_feedback):
            camb.set_feedback_level(level=3)
        else:
            camb.set_feedback_level(level=0)

        if (high_precision):
            cp[i][j].set_accuracy(
                            DoLateRadTruncation=False,
                            AccuracyBoost=5,
                            lAccuracyBoost=5,
                            lSampleBoost=5,
                            )
        else:
            cp[i][j].set_accuracy(
                            DoLateRadTruncation=False,
                            AccuracyBoost=1,
                            lAccuracyBoost=1,
                            )    
        start = time.time()
        CambRun[i][j] = camb.get_results(cp[i][j])
        print("CAMB run %d-%d took %f seconds"%(i, j, time.time()-start))


In [ ]:
# Per-cosmology spectra of the sample on kvec_sample: computed from the CAMB runs above (RECOMPUTE["camb"]) or reloaded from data/.
_S = CFG['suffix']
_files = {'Pkl_hmc': 'data/Pklin/Pklin_%s_%%d.txt' % _S, 'Pk_hmc': 'data/PkHM/PkHM_%s_%%d.txt' % _S,
          'Pk_whmc_hf_tweaked': 'data/PkWHMhf/PkWHMhftweaked_%s_%%d.txt' % _S}
if SAMPLE == 'BACCO':
    _files.update({'Pk_ori': 'data/PkHMori/PkHMori_%s_%%d.txt' % _S,
                   'Pk_whmc_hf_tweaked_warren': 'data/PkWHMhf/PkWHMhftweaked_warren_%s_%%d.txt' % _S})
_case = {'Pk_hmc': 0, 'Pk_ori': 1, 'Pk_whmc_hf_tweaked': 2, 'Pk_whmc_hf_tweaked_warren': 3} if SAMPLE == 'BACCO' else {'Pk_hmc': 0, 'Pk_whmc_hf_tweaked': 1}
_spec = {name: np.zeros((Ncosmo,len(redshifts),len(kvec_sample))) for name in _files}
for i in range(0,Ncosmo):
    if RECOMPUTE["camb"]:
        _PKL = CambRun[i][0].get_matter_power_interpolator(nonlinear=False, hubble_units=hubble_units_sample, k_hunit=hubble_units_sample)
        _PK = {name: CambRun[i][j].get_matter_power_interpolator(nonlinear=True, hubble_units=hubble_units_sample, k_hunit=hubble_units_sample) for name, j in _case.items()}
        for iz in range(0,len(redshifts)):
            _spec['Pkl_hmc'][i][iz] = [_PKL.P(redshifts[iz],k) for k in kvec_sample]
            for name in _case: _spec[name][i][iz] = [_PK[name].P(redshifts[iz],k) for k in kvec_sample]
    else:
        for name, f in _files.items():
            for iz in range(0,len(redshifts)):
                _spec[name][i][iz] = np.loadtxt(f % i, usecols=(iz+1))
Pkl_hmc = _spec['Pkl_hmc']; Pk_hmc = _spec['Pk_hmc']; Pk_whmc_hf_tweaked = _spec['Pk_whmc_hf_tweaked']
if SAMPLE == 'BACCO':
    Pk_ori = _spec['Pk_ori']; Pk_whmc_hf_tweaked_warren = _spec['Pk_whmc_hf_tweaked_warren']


In [ ]:
if WRITE["camb"]:
    for name, f in _files.items():
        for i in range(0,Ncosmo):
            np.savetxt(f % i, np.c_[kvec_sample, _spec[name][i][0], _spec[name][i][1], _spec[name][i][2]])


In [ ]:
# 1-loop LPT (velocileptors) for the sample cosmologies, k_IR = k_cutoff = f * sqrt(5) * k_nl(z) with f = 1 as the baseline
# (Figs. 5, 6). For the BACCO sample the cutoff variants of Fig. A2 are computed as well: 'def' = fixed 10 Mpc^-1 and
# 'defscaled' = 10 Mpc^-1 scaled by k_nl(z)/k_nl,fid(z) (the variant plotted in the lower panel of Fig. A2).
# Also tabulates k_nl(z) and 1/sigma_v(z) of the sample and of the fiducial cosmology (read by the LPT-scatter notebook).
LPT_variants = [1.0] + ([0.5, 2.0, 5.0, 'def', 'defscaled'] if SAMPLE == 'BACCO' else [])
knl = np.zeros((Ncosmo,len(redshifts))); kdisp = np.zeros((Ncosmo,len(redshifts)))
knl_fid = np.zeros(len(redshifts)); kdisp_fid = np.zeros(len(redshifts))
for iz in range(0,len(redshifts)):
    knl_fid[iz] = M_fid.nonlinear_scale(redshifts, len(redshifts))[iz]
    kdisp_fid[iz] = 1.0/M_fid.fourier_hmcode_sigmadisp(redshifts, len(redshifts))[iz]
    for i in range(0,Ncosmo):
        knl[i,iz] = M[i].nonlinear_scale(redshifts, len(redshifts))[iz]
        kdisp[i,iz] = 1.0/M[i].fourier_hmcode_sigmadisp(redshifts, len(redshifts))[iz]
if WRITE["camb"]:
    np.savetxt('data/knl/knl_fid.txt', knl_fid); np.savetxt('data/knl/kv_fid.txt', kdisp_fid)
    np.savetxt('data/knl/knl_%s.txt' % CFG['suffix'], knl); np.savetxt('data/knl/kv_%s.txt' % CFG['suffix'], kdisp)
Pk_lpt = {}
for f in LPT_variants:
    _ftag = f if isinstance(f, str) else '%.1f' % f
    Pk_lpt[f] = np.zeros((Ncosmo,len(redshifts),len(kvec_sample)))
    for i in range(0,Ncosmo):
        _fname = 'data/PkLPT/PkLPT_kNL%s_%s_%d.txt' % (_ftag, CFG['suffix'], i)
        if (not RECOMPUTE["velocileptors"]) and os.path.isfile(_fname):
            for iz in range(0,len(redshifts)):
                Pk_lpt[f][i][iz] = np.nan_to_num(np.loadtxt(_fname, usecols=(iz+1)))
        else:
            _h = M[i].h() if hubble_units_sample else 1.0
            for iz in range(0,len(redshifts)):
                _kc = 10.0 if f == 'def' else (10.0*knl[i,iz]/knl_fid[iz] if f == 'defscaled' else f*np.sqrt(5)*knl[i,iz])
                lpt = LPT_RSD(kvec_sample*_h, Pkl_hmc[i][iz]/_h**3., kIR=_kc, cutoff=_kc, jn=15, one_loop=True, **LPT_GRID)
                dummy, _p, dummy, dummy = lpt.make_pell_fixedbias(0, np.zeros(11), kv=kvec_sample*_h)
                Pk_lpt[f][i][iz] = np.nan_to_num(_p*_h**3.)
            if WRITE["velocileptors"]:
                np.savetxt(_fname, np.c_[kvec_sample, Pk_lpt[f][i][0], Pk_lpt[f][i][1], Pk_lpt[f][i][2]])


In [ ]:
# esthetic definitions for the plots
import os.path
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
font = {'size'   : 16, 'family':'STIXGeneral'}
axislabelfontsize='large'
matplotlib.rc('font', **font)
matplotlib.rc('text', usetex=True)
plt.rcParams['legend.fontsize']='medium'
plt.rcParams['axes.linewidth'] = 2
plt.rcParams["figure.figsize"] = [16.0,15.0]

if SAMPLE != 'BACCO':
    print('Fig. 5 is made for the BACCO sample (needs the Warren-HMF and HMcode-original runs); skipped for', SAMPLE)
else:
    # Fig. 5: 1-loop LPT, LPT + halo model, LPT + WHM (Sheth-Tormen HMF), LPT + WHM (Warren HMF) and HMcode-2020 against the
    # reference spectra of the sample (paper: BACCO sample). All per-cosmology inputs come from the cells above / data/.
    cs = [4.0, 1.2, 0.3]
    rows = 5
    txtfs = 16
    cmap = plt.get_cmap('viridis')
    variable = s8
    vmin = 0.7
    vmax = 0.9
    norm = Normalize(vmin=vmin, vmax=vmax)
    sm = ScalarMappable(norm=norm, cmap=cmap)
    fig_Pk, ax_Pk = plt.subplots(rows, len(redshifts), sharex=True, sharey=True, gridspec_kw={'hspace': 0.0, 'wspace':0.0})
    for i in range(0,Ncosmo):
        for iz in range(0,len(redshifts)):
            _ref = pk_nl_ref[i][iz]; _lpt = Pk_lpt[1.0][i][iz]
            color1 = cmap(norm(variable[i]))
            ax_Pk[0,iz].semilogx(kvec_sample, 100*(_lpt-_ref)/_ref, linestyle=redshift_styles[iz],c=color1)
            ax_Pk[1,iz].semilogx(kvec_sample, 100*(Pk_ori[i][iz]-_ref-Pkl_hmc[i][iz]+_lpt)/_ref, linestyle=redshift_styles[iz], color=color1)
            ax_Pk[2,iz].semilogx(kvec_sample, 100*(Pk_whmc_hf_tweaked[i][iz]-_ref-Pkl_hmc[i][iz]+_lpt)/_ref, linestyle=redshift_styles[iz], color=color1)
            ax_Pk[3,iz].semilogx(kvec_sample, 100*(Pk_whmc_hf_tweaked_warren[i][iz]-_ref-Pkl_hmc[i][iz]+_lpt)/_ref, linestyle=redshift_styles[iz], color=color1)
            ax_Pk[4,iz].semilogx(kvec_sample, 100*(Pk_hmc[i][iz]-_ref)/_ref, linestyle=redshift_styles[iz], color=color1)
            if (i==0):
                if (iz==0):
                    for _row, textstr in enumerate([r'\bf $\bm{1\ell}$-LPT', r'\bf $\bm{1\ell}$-LPT + HM', r'\bf $\bm{1\ell}$-LPT + WHM (baseline)',
                                                    r'\bf $\bm{1\ell}$-LPT + WHM (Warren)', r'\bf HMcode-2020']):
                        ax_Pk[_row,iz].text(0.05, 0.05, textstr, transform=ax_Pk[_row,iz].transAxes, fontsize=txtfs, verticalalignment='bottom',ha='left')
                textstr = r'\bf z=%.1f'%(redshifts[iz])
                ax_Pk[0,iz].text(0.95, 0.95, textstr, transform=ax_Pk[0,iz].transAxes, fontsize=txtfs, verticalalignment='top',ha='right')
                for jcase in range(0,rows):
                    ax_Pk[jcase,iz].tick_params(which='major', top=True, left=True, right=True, bottom=True, direction='in', length=8, width=1)
                    ax_Pk[jcase,iz].tick_params(which='minor', top=True, left=True, right=True, bottom=True, direction='in', length=4)
                    ax_Pk[jcase,iz].tick_params(axis='x', bottom=True, direction='inout')
                    ax_Pk[jcase,iz].tick_params(axis='both', labelsize=16)
                    ax_Pk[jcase,iz].fill_between(kvec, 100*(response_low[iz]-1), 100*(response_high[iz]-1), alpha=0.2, color='darkorange')
    textstr = r'$\sigma_\mathrm{v}^{-1}(z)$'
    ax_Pk[0,0].text((np.log10(kdisp_fid[0]/M_fid.h())-np.log10(0.01))/(np.log10(4.8)-np.log10(0.01))-0.01, 0.95, textstr, transform=ax_Pk[0,0].transAxes, fontsize=14, c='magenta',
            verticalalignment='top',ha='right')#, bbox=dict(alpha=0.5))

    textstr = r'$r_\mathrm{nl}^{-1}(z)$'
    ax_Pk[0,0].text((np.log10(knl_fid[0]/M_fid.h())-np.log10(0.01))/(np.log10(4.8)-np.log10(0.01))+0.01, 0.95, textstr, transform=ax_Pk[0,0].transAxes, fontsize=14, c='magenta',
            verticalalignment='top',ha='left')#, bbox=dict(alpha=0.5))

    for jcase in range(0,rows):
        for iz in range(0,len(redshifts)):
            ax_Pk[jcase,iz].axvline(kdisp_fid[iz]/M_fid.h(),c='magenta',ls=':')
            ax_Pk[jcase,iz].axvline(knl_fid[iz]/M_fid.h(),c='magenta',ls='--')
            ax_Pk[jcase,iz].axhline(0.0,c='k',alpha=0.5)
            ax_Pk[jcase,iz].axhline(3,c='k',ls=':',alpha=0.5)
            ax_Pk[jcase,iz].axhline(-3,c='k',ls=':',alpha=0.5)
            ax_Pk[jcase,iz].axhline(5,c='k',ls='--',alpha=0.5)
            ax_Pk[jcase,iz].axhline(-5,c='k',ls='--',alpha=0.5)

    axixfs = 18            
    ax_Pk[0,0].set_xlim([0.01,5])
    ax_Pk[0,0].set_ylim([-22,12])
    ax_Pk[4,1].set_xlabel(r'$k \,[h\mathrm{Mpc}^{-1}]$', fontsize=24,labelpad=0)
    ax_Pk[2,0].set_ylabel(r'$ \frac{P^\mathrm{X}(k)-P^\mathrm{NL}(k)}{P^\mathrm{NL}(k)}$', fontsize=28,labelpad=0)

    # Add a colorbar with equidistant ticks
    cbar = fig_Pk.colorbar(sm, ax=ax_Pk, location='right', shrink=0.6, aspect=30)
    cbar.set_label(r'$\sigma_8$',fontsize=24)
    cbar.set_ticks(np.linspace(vmin, vmax, 5))  # 5 equidistant ticks

    plt.show()  
    _figname = 'w0waMnu_LPT_kNL1.0_BACCO_baselineaccuratebf_concnfwwarren_100.png' if SAMPLE == 'BACCO' else 'w0waMnu_LPT_kNL1.0_%s_100.png' % SAMPLE
    fig_Pk.savefig(os.path.join(FIG_DIR, _figname),bbox_inches='tight') 
 
